# Phase 7 — LLM-First Generation + Retrieval Grounding (European LLM)

완전히 다른 접근. retrieval 주도 → **LLM 주도**.

## 왜 바꿨나

이전 phases 의 retrieval 은 **실체법**(살인→StGB)은 잡지만 **절차법**(BGG 항소, StPO 절차, BV 청문권)을 못 잡음. bar exam 답 25개의 상당수가 절차법 → fact-similarity 로 불가능, **법적 추론** 필요.

LB 기록: phase1(seeds만) 0.08 > phase6(retrieval+anchor) 0.056. 복잡한 retrieval 이 noise 만 추가.

## 새 파이프라인

```
사건(영어)
  → regex seeds (고정밀, 쿼리에 적힌 citation)
  → retrieval grounding: 캐시된 BGE-M3 로 관련 법조문 텍스트 top-50 제공
  → 유럽 LLM 이 사건+grounding 읽고 실체법+절차법 citation 추론 생성
  → 정규화 + corpus exact-match 검증 (환각 자동 제거)
  → seeds + universal default + 검증된 LLM citation → emit
```

## 모델 — 유럽 LLM (독/프/이 강함)

`USE_LLM_MODEL` 자동 발견: Mistral-Small-22B / Mixtral-8x7B / Mistral-Nemo-12B / EuroLLM-9B / Teuken-7B 중 attach 된 것.

## 캐시 재사용

phase 6 의 `/kaggle/working/cache_phase3/` (laws/court FAISS, court_meta) 그대로 사용 → 6시간 인코딩 안 함, ~40분.

## Settings
- GPU T4×2, Internet ON (LLM HF fallback), Persistence ON (캐시)
- Add Input: bge-m3 + 유럽 LLM (Models 탭)


In [ ]:
# Cell 1 — env, paths, pip
!pip install -q rank_bm25 sentence-transformers faiss-cpu

import re, csv, json, pickle, time, gc, sys, os
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

DATA = None
for cand in [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]:
    if cand.exists() and (cand / 'laws_de.csv').exists():
        DATA = cand; break
if DATA is None:
    raise RuntimeError(f'Data not found. /kaggle/input contents: {list(Path("/kaggle/input").iterdir())}')

WORK = Path('/kaggle/working')
CACHE = WORK / 'cache_phase3'
CACHE.mkdir(parents=True, exist_ok=True)
print(f'DATA: {DATA}')
print(f'WORK: {WORK}')

for p in ['train.csv', 'val.csv', 'test.csv', 'laws_de.csv', 'court_considerations.csv']:
    fp = DATA / p
    print(f'  {p:30s} {"ok" if fp.exists() else "MISSING"}  ({fp.stat().st_size/1e6:.1f} MB)' if fp.exists() else f'  {p:30s} MISSING')

# Toggles
USE_TRANSLATE = True       # NLLB EN→DE
USE_RERANKER  = True       # BGE-reranker-v2-m3
USE_COURT     = True       # court_considerations 2-stage (most expensive)
USE_LLM       = False      # Qwen verify — off by default; set True only if you've attached the model + have time


In [ ]:
import json

# Cell 2 — Multilingual abbreviation map (FR/IT → DE) — 1662 entries from Omnilex starter repo
MULTILANG_ABBR = {'OIAA': 'AAMV', 'OMAA': 'HVUV', 'DE-OMBat': 'AB-VASm', 'OBE-FINMA': 'ABV-FINMA', 'OAdo': 'AdoV', 'OAdoz': 'AdoV', 'SDA': 'ADS', 'ORAT': 'AEFV', 'OIAgr': 'AEV', 'LMCFA': 'AFZFG', 'LMCCE': 'AFZFG', 'OMCFA': 'AFZFV', 'OMCCE': 'AFZFV', 'LAVS': 'AHVG', 'RAVS': 'AHVV', 'OAVS': 'AHVV', 'LEAR': 'AIAG', 'LSAI': 'AIAG', 'OEAR': 'AIAV', 'OSAIn': 'AIAV', 'LEI': 'AIG', 'LStrI': 'AIG', 'OAccD': 'AkkBV', 'AccredO-LPsy': 'AkkredV-PsyG', 'OEAc-LPPsi': 'AkkredV-PsyG', 'LEDPP': 'ALBAG', 'LSRPP': 'ALBAG', 'OEDPP': 'ALBAV', 'OSRPP': 'ALBAV', 'OBat': 'AlgV', 'OSRA': 'V-NDA', 'OdA': 'AlkBestV', 'OTAl': 'AlkBestV', 'LAlc': 'AlkG', 'OAlc': 'AlkV', 'OAllerg': 'AllergV', 'OGEmol': 'AllgGebV', 'OgeEm': 'AllgGebV', 'OSites': 'AltlV', 'OSiti': 'AltlV', 'OSI-AC': 'ALV-IsV', 'OAMéd': 'AMBV', 'OAMed': 'AMBV', 'OEMéd': 'AMZV', 'OOMed': 'AMZV', 'OOrgA': 'AO', 'OEs': 'AO', 'OOS': 'AOV', 'OOV': 'AOV', 'LTr': 'ArG', 'LL': 'ArG', 'OLT 1': 'ArGV 1', 'OLL 1': 'ArGV 1', 'OLT 2': 'ArGV 2', 'OLL 2': 'ArGV 2', 'OLT 3': 'ArGV 3', 'OLL 3': 'ArGV 3', 'OLT 4': 'ArGV 4', 'OLL 4': 'ArGV 4', 'OLT 5': 'ArGV 5', 'OLL 5': 'ArGV 5', 'OITRV': 'ARPV', 'OTR 1': 'ARV 1', 'OLR 1': 'ARV 1', 'OTR 2': 'ARV 2', 'OLR 2': 'ARV 2', 'LSEtr': 'ASG', 'LSEst': 'ASG', 'Limpauto': 'AStG', 'LIAut': 'AStG', 'Oimpauto': 'AStV', 'OIAut': 'AStV', 'OSur-ASR': 'ASV-RAB', 'OS-ASR': 'ASV-RAB', 'LAsi': 'AsylG', 'OA 1': 'AsylV 1', 'OAsi 1': 'AsylV 1', 'OA 2': 'AsylV 2', 'OAsi 2': 'AsylV 2', 'OA 3': 'AsylV 3', 'OAsi 3': 'AsylV 3', 'LTrAlp': 'AtraG', 'LTAlp': 'AtraG', 'Otransa': 'AtraV', 'OTrAl': 'AtraV', 'LPGA': 'ATSG', 'OPGA': 'ATSV', 'RSTF': 'AufRBGer', 'RVTF': 'AufRBGer', 'OAsc': 'AufzV', 'OSAC': 'AuLaV', 'OAEs': 'AuLaV', 'LECCT': 'AVEG', 'LOCCL': 'AVEG', 'OFAC': 'AVFV', 'OFAD': 'AVFV', 'LSE': 'AVG', 'LC': 'AVG', 'LACI': 'AVIG', 'LADI': 'AVIG', 'OACI': 'AVIV', 'OADI': 'AVIV', 'OS': 'AVO', 'OS-FINMA': 'AVO-FINMA', 'OSE': 'AVV', 'OC': 'AVV', 'LDI': 'AwG', 'LDT': 'AZG', 'LDL': 'AZG', 'OLDT': 'AZGV', 'OLDL': 'AZGV', 'ODMA': 'BAlV', 'LB': 'BankG', 'LBCR': 'BankG', 'OB': 'BankV', 'OBCR': 'BankV', 'OTConst': 'BauAV', 'OLCostr': 'BauAV', 'LPCo': 'BauPG', 'LProdC': 'BauPG', 'OPCo': 'BauPV', 'OProdC': 'BauPV', 'LFPr': 'BBG', 'OFPr': 'BBV', 'LTI': 'BEG', 'LTCo': 'BEG', 'LHand': 'BehiG', 'LDis': 'BehiG', 'OHand': 'BehiV', 'ODis': 'BehiV', 'ONo-ASR': 'BekV-RAB', 'OC-ASR': 'BekV-RAB', 'LStup': 'BetmG', 'OCStup': 'BetmKV', 'OEPStup': 'BetmPV', 'OSPStup': 'BetmPV', 'OAStup': 'BetmSV', 'ODStup': 'BetmSV', 'OTStup-DFI': 'BetmVV-EDI', 'OEStup-DFI': 'BetmVV-EDI', 'OProP': 'BevSV', 'OPPop': 'BevSV', 'LFAIE': 'BewG', 'LAFE': 'BewG', 'OAIE': 'BewV', 'OAFE': 'BewV', 'LF-CLaH': 'BG-HAÜ', 'LF-CAA': 'BG-HAÜ', 'LTEJUS': 'BG-RVUS', 'LTAGSU': 'BG-RVUS', 'LAr': 'BGA', 'LDFR': 'BGBB', 'LMI': 'BGBM', 'LCITES': 'BGCITES', 'LF-CITES': 'BGCITES', 'RTF': 'BGerR', 'LFSP': 'BGF', 'LLCA': 'BGFA', 'LTF': 'BGG', 'LDEA': 'BGIAA', 'LSISA': 'BGIAA', 'LBCF': 'BGLE', 'LRFF': 'BGLE', 'LPPS': 'BGMD', 'LDPS': 'BGMD', 'LFPC': 'BGMK', 'LTrans': 'BGÖ', 'LTras': 'BGÖ', 'LEAC': 'BGRB', 'LIAC': 'BGRB', 'LJAr': 'BGS', 'LGD': 'BGS', 'LTN': 'BGSA', 'LLN': 'BGSA', 'LOST': 'BGST', 'LFSI': 'BGST', 'LFIF': 'BIFG', 'OBiG': 'BiGV', 'OIB-FINMA': 'BIV-FINMA', 'LCESF': 'BiZG', 'LCSFS': 'BiZG', 'LCMIF': 'BIZMB', 'OMPr': 'BMV', 'LMP': 'BöB', 'LAPub': 'BöB', 'OPDC': 'BPDV', 'OPDPers': 'BPDV', 'LSIP': 'BPI', 'LDP': 'BPR', 'LPSP': 'BPS', 'RNC': 'BSO', 'LSF': 'BStatG', 'LStat': 'BStatG', 'LIB': 'BStG', 'RAATPF': 'BStGerNR', 'ROATPF': 'BStGerNR', 'ROTPF': 'BStGerOR', 'RFPPF': 'BStKR', 'RSPPF': 'BStKR', 'OBioc': 'VBP', 'OBcarb': 'BTrV', 'LN': 'BüG', 'LCit': 'BüG', 'LSCPT': 'BÜPF', 'OREE': 'BURV', 'ORIS': 'BURV', 'OLN': 'VOSA', 'OCit': 'BüV', 'Cst.': 'BV', 'Cost.': 'BV', 'LPP': 'BVG', 'OPP 1': 'BVV 1', 'OPP 2': 'BVV 2', 'OPP 3': 'BVV 3', 'LIDV': 'BVVG', 'LDDV': 'BVVG', 'LMSI': 'BWIS', 'PCF': 'BZP', 'PC': 'BZP', 'OCart': 'CartV', 'LChim': 'ChemG', 'LPChim': 'ChemG', 'OEChim': 'ChemGebV', 'OEPChim': 'ChemGebV', 'OPICChim': 'ChemPICV', 'ORRChim': 'ChemRRV', 'ORRPChim': 'ChemRRV', 'OChim': 'ChemV', 'OPChim': 'ChemV', 'OCPCh': 'ChKV', 'OCCC': 'ChKV', 'CRTIF': 'COTIF 1980', 'LCaS-COVID-19': 'Covid-19-SBüG', 'LFiS-COVID-19': 'Covid-19-SBüG', 'OCyS': 'CSV', 'OCS': 'GGBV', 'CAC': 'CWÜ', 'OACP': 'CZV', 'OAut': 'CZV', 'ACCEES': 'ASEEGKVBSPMSA', 'ACSCSSS': 'ASEEGKVBSPMSA', 'LIFD': 'DBG', 'OSRP': 'DBV', 'OTDD': 'DBZV', 'LDes': 'DesG', 'ODes': 'DesV', 'OSEP': 'DGV', 'OSAP': 'DGV', 'RSA': 'DRA', 'RSE': 'DRA', 'LPD': 'DSG', 'OEng': 'DüV', 'OCon': 'DüV', 'OD-ASR': 'DV-RAB', 'OPD': 'DZV', 'LCdF': 'EBG', 'Lferr': 'EBG', 'OCF': 'EBV', 'Oferr': 'EBV', 'OITE-PT': 'EDAV-DS', 'OITE-PT-DFI': 'EDAV-DS-EDI', 'OITE-UE': 'EDAV-EU', 'OITE-UE-DFI': 'EDAV-EU-EDI', 'OITE-AC': 'EDAV-Ht', 'OITEAc': 'EDAV-Ht', 'OEES': 'EESV', 'O-HEFSM': 'EHSM-V', 'O-SUFSM': 'EHSM-V', 'OEmV': 'EichGebV', 'OEm-V': 'EichGebV', 'OIPR': 'EigVV', 'RIPP': 'EigVV', 'LIFM': 'EIMG', 'OIFM': 'EIMV', 'OO': 'EiV', 'OU': 'EiV', 'OCCP': 'EKBV', 'OCSC': 'EKBV', 'OESC-OFAG': 'EKV-BLW', 'OVC-UFAG': 'EKV-BLW', 'LIE': 'EleG', 'LPC': 'ELG', 'OPC-AVS/AI': 'ELV', 'LMETA': 'EMBAG', 'LMeCA': 'EMBAG', 'OMETA': 'EMBAV', 'OMeCA': 'EMBAV', 'LEmb': 'EmbG', 'OESMB': 'EmGvV', 'OACMB': 'EmGvV', 'LCMP': 'EMKG', 'OCMP': 'EMKV', 'OIMepe': 'EMmV', 'OSMisE': 'EMmV', 'CSDLLF': 'KSMG', 'CSDDDLF': 'KSMG', 'OEEE': 'EnEV', 'OEEne': 'EnEV', 'OEneR': 'EnFV', 'OPEn': 'EnFV', 'LIFSN': 'ENSIG', 'OIFSN': 'ENSIV', 'LDét': 'EntsG', 'LDist': 'EntsG', 'Odét': 'EntsV', 'ODist': 'EntsV', 'OAAE': 'EÖBV', 'OAPuE': 'EÖBV', 'OAAE-DFJP': 'EÖBV-EJPD', 'OAPuE-DFGP': 'EÖBV-EJPD', 'LAPG': 'EOG', 'LIPG': 'EOG', 'OAPG': 'EOV', 'OIPG': 'EOV', 'OFDEP': 'EPDFV', 'OFCIP': 'EPDFV', 'LDEP': 'EPDG', 'LCIP': 'EPDG', 'ODEP': 'EPDV', 'OCIP': 'EPDV', 'ODEP-DFI': 'EPDV-EDI', 'OCIP-DFI': 'EPDV-EDI', 'LEp': 'EpG', 'CBE 2000': 'EPÜ 2000', 'OEp': 'EpV', 'OFR': 'ERV', 'OFoP': 'ERV', 'CE-TAF': 'ERV-BVGer', 'OUC': 'ESV', 'OIConf': 'ESV', 'Oexpa': 'ExpaV', 'Oespa': 'ExpaV', 'LAFam': 'FamZG', 'OAFam': 'FamZV', 'OAFami': 'FamZV', 'OEV 4': 'FAV 4', 'OEA 4': 'FAV 4', 'OSVo': 'FDO', 'LSFin': 'FIDLEG', 'LSerFi': 'FIDLEG', 'OSFin': 'FIDLEV', 'OSerFi': 'FIDLEV', 'LERI': 'FIFG', 'LPRI': 'FIFG', 'OECin': 'FiFV', 'OPCin': 'FiFV', 'LIMF': 'FinfraG', 'LInFi': 'FinfraG', 'OIMF': 'FinfraV', 'OInFi': 'FinfraV', 'OIMF-FINMA': 'FinfraV-FINMA', 'OInFi-FINMA': 'FinfraV-FINMA', 'LEFin': 'FINIG', 'LIsFi': 'FINIG', 'OEFin': 'FINIV', 'OIsFi': 'FINIV', 'OEFin-FINMA': 'FINIV-FINMA', 'OIsFi-FINMA': 'FINIV-FINMA', 'Oém-FINMA': 'FINMA-GebV', 'Oem-FINMA': 'FINMA-GebV', 'OA-FINMA': 'FINMA-PV', 'LFINMA': 'FINMAG', 'OMPRI': 'FIPBV', 'LFiEl': 'FiREG', 'LAiSE': 'FiREG', 'CRSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'CSSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'LCF': 'FKG', 'OReCI': 'FKINV', 'OPRCI': 'FKINV', 'LFA': 'FLG', 'LAF': 'FLG', 'RFA': 'FLV', 'OA Fam': 'FLV', 'OLALA': 'FMBV', 'OLAlA': 'FMBV', 'LPMA': 'FMedG', 'LPAM': 'FMedG', 'OPMA': 'VLIp', 'OMP': 'VöB', 'LTC': 'FMG', 'OSALA': 'FMV', 'OsAlA': 'FMV', 'OFOrg': 'FOrgV', 'OAOrg': 'FOrgV', 'OH': 'FPV', 'OOra': 'FPV', 'OQICin': 'FQIV', 'OQIC': 'FQIV', 'ODE': 'FrSV', 'OEDA': 'FrSV', 'LFus': 'FusG', 'OMCo': 'FV', 'OMaeC': 'FV', 'OF-SCPT': 'FV-ÜPF', 'AECSDPC': 'ASEEGMF', 'ACSCS': 'ASEEGMF', 'OAG': 'GaGV', 'OAppG': 'GaGV', 'AGTDC': 'GATT', 'ORF': 'GBV', 'REmol-TAF': 'GebR-BVGer', 'TA-TAF': 'GebR-BVGer', 'REmol-TFB': 'GebR-PatGer', 'TA-TFB': 'GebR-PatGer', 'Olico': 'GeBüV', 'Olc': 'GeBüV', 'OE ArchF': 'GebV BAR', 'OE ARF': 'GebV BAR', 'OEmol-AFC': 'GebV ESTV', 'OEm-AFC': 'GebV ESTV', 'OELP': 'GebV SchKG', 'OTLEF': 'GebV SchKG', 'Oem-LEI': 'GebV-AIG', 'OEmol-LStrI': 'GebV-AIG', 'Oem-Acc': 'GebV-Akk', 'Oemo-Acc': 'GebV-Akk', 'OEmol-LTr': 'GebV-ArG', 'OEm-LL': 'GebV-ArG', 'OEmol-OFRO': 'GebV-ASTRA', 'OEmo-USTRA': 'GebV-ASTRA', 'OEmol-LSE': 'GebV-AVG', 'OEm-LC': 'GebV-AVG', 'OEmol-OFEV': 'GebV-BAFU', 'OE-UFAM': 'GebV-BAFU', 'OEmol-OFSPO': 'GebV-BASPO', 'OEm UFSPO': 'GebV-BASPO', 'OEmol-OFAC': 'GebV-BAZL', 'OEm-UFAC': 'GebV-BAZL', 'Oem-OFJ': 'GebV-BJ', 'Oem UFG': 'GebV-BJ', 'OEmol-OFAG': 'GebV-BLW', 'Ordinanza sulle tasse UFAG': 'GebV-BLW', 'OEmol-DFAE': 'GebV-EDA', 'OEm-DFAE': 'GebV-EDA', 'OEmol-DFI-BN': 'GebV-EDI-NBib', 'OEm-DFI-BN': 'GebV-EDI-NBib', 'OEmol-CMP': 'GebV-EMK', 'OEm-CMP': 'GebV-EMK', 'Oémol-En': 'GebV-En', 'OE-En': 'GebV-En', 'OEmol-ASF': 'GebV-ESA', 'OEm-AVF': 'GebV-ESA', 'OEmol-fedpol': 'GebV-fedpol', 'OEm-fedpol': 'GebV-fedpol', 'OREDT': 'GebV-FMG', 'OTST': 'GebV-FMG', 'OEmol-RC': 'GebV-HReg', 'OTa-IPI': 'GebV-IGE', 'OEmol-LCart': 'GebV-KG', 'OEm-LCart': 'GebV-KG', 'OEm-METAS': 'GebV-METAS', 'Oem-METAS': 'GebV-METAS', 'OEmol-BN': 'GebV-NBib', 'OEm-BN': 'GebV-NBib', 'OEmol-TP': 'GebV-öV', 'OEm-TP': 'GebV-öV', 'OEmol-Publ': 'GebV-Publ', 'OEm-Pub': 'GebV-Publ', 'OÉmol-CSA': 'GebV-SAR', 'OEm-CSA': 'GebV-SAR', 'OEmol-SEFRI': 'GebV-SBFI', 'OEm-SEFRI': 'GebV-SBFI', 'OE-RaP': 'GebV-StS', 'OEm-RaP': 'GebV-StS', 'OEmol-OHB': 'GebV-TPS', 'OEmSON': 'GebV-TPS', 'OEmol-DDPS': 'GebV-VBS', 'OEm-DDPS': 'GebV-VBS', 'OÉIPPSS': 'GebVO St', 'OSEIPSS': 'GebVO St', 'LGéo': 'GeoIG', 'LGI': 'GeoIG', 'OGéo': 'GeoIV', 'OGI': 'GeoIV', 'OGéo-swisstopo': 'GeoIV-swisstopo', 'OGI-swisstopo': 'GeoIV-swisstopo', 'OGéom': 'GeomV', 'Ogeom': 'GeomV', 'ONGéo': 'GeoNV', 'ONGeo': 'GeoNV', 'ORPSan': 'GesBAV', 'LPSan': 'GesBG', 'OCPSan': 'GesBKV', 'OSAS': 'GGBV', 'OCMD': 'GGUV', 'OMCont': 'GGUV', 'OIC-DFI': 'GgV-EDI', 'LCB': 'PAG', 'LBDI': 'GKG', 'ODVo': 'GKZ', 'OCPo': 'GKZ', 'LEg': 'GlG', 'LPar': 'GlG', 'OBPL': 'GLPV', 'RTFB': 'GR-PatGer', 'RI-COMCO': 'GR-WEKO', 'RCN': 'GRN', 'RCE': 'GRS', 'RCS': 'GRS', 'LEaux': 'GSchG', 'LPAc': 'GSchG', 'OEaux': 'GSchV', 'OPAc': 'GSchV', 'LGG': 'GTG', 'LIG': 'GTG', 'LAGH': 'GUMG', 'LEGU': 'GUMG', 'OAGH': 'GUMV', 'OEGU': 'GUMV', 'LTM': 'GüTG', 'OTM': 'GüTV', 'LTTM': 'GVVG', 'LTrasf': 'GVVG', 'LBA': 'GwG', 'LRD': 'GwG', 'OBA': 'GwV', 'ORD': 'GwV', 'OBA-OFDF': 'GwV-BAZG', 'ORD-UDSC': 'GwV-BAZG', 'OBA-DFJP': 'GwV-EJPD', 'ORD-DFGP': 'GwV-EJPD', 'OBA-CFMJ': 'GwV-ESBK', 'ORD-CFCG': 'GwV-ESBK', 'OBA-FINMA': 'GwV-FINMA', 'ORD-FINMA': 'GwV-FINMA', 'LTrD': 'HArG', 'LLD': 'HArG', 'OTrD': 'HArGV', 'OLLD': 'HArGV', 'OIPSD': 'HasLV', 'OIPSDA': 'HasLV', 'OIPSD-DEFR': 'HasLV-WBF', 'OIPSDA-DEFR': 'HasLV-WBF', 'OPFP-FINMA': 'HBEV-FINMA', 'LRH': 'HFG', 'LRUm': 'HFG', 'LEHE': 'HFKG', 'LPSU': 'HFKG', 'OMCR 20': 'HFMV 20', 'OPCR 20': 'HFMV 20', 'OMCR 22': 'HFMV 22', 'OPCR 22': 'HFMV 22', 'ORH': 'HFV', 'ORUm': 'HFV', 'LLGV': 'HGVAnG', 'LRAV': 'HGVAnG', 'OCBo': 'HHV', 'OCoL': 'HHV', 'CLaH96': 'HKsÜ', 'Convenzione dell’Aia sulla protezione dei minori': 'HKsÜ', 'OGOM': 'HKSV', 'OGOE': 'HKSV', 'LPTh': 'HMG', 'LATer': 'HMG', 'ORC': 'HRegV', 'OCCHE': 'HSBBV', 'OSCU': 'HSBBV', 'OMAV': 'HVA', 'OMAI': 'HVI', 'OMAINF': 'HVUV', 'OHyg': 'HyV', 'ORI': 'HyV', 'OIAM': 'IAMV', 'OSFPrHE': 'IBH-V', 'O-SIFPU': 'IBH-V', 'LSIS': 'SIaG', 'LSISpo': 'IBSG', 'OSIS': 'IBSV', 'OSISpo': 'IBSV', 'OMCC': 'VSFK', 'OCoCr': 'IBTV', 'OId-BDTA': 'IdTVD-V', 'OIBDTA': 'IdTVD-V', 'LIPPI': 'IFEG', 'LIPIn': 'IFEG', 'OIPI': 'IGE-OV', 'Oper-IPI': 'IGE-PersV', 'OPer-IPI': 'IGE-PersV', 'LIPI': 'IGEG', 'OAiR': 'InkHV', 'OAInc': 'InkHV', 'Oinv': 'InvV', 'OPICin': 'IPFiV', 'LDIP': 'IPRG', 'LISint': 'IQG', 'LIFI': 'IQG', 'RInfo-TFB': 'IR-PatGer', 'EIMP': 'IRSG', 'AIMP': 'IRSG', 'OEIMP': 'IRSV', 'OAIMP': 'IRSV', 'O-SI ABV': 'ISABV-V', 'O-SIAMV': 'ISABV-V', 'LSI': 'ISG', 'LSIn': 'ISG', 'O-SICAL': 'ISLK-V', 'O-SIFA': 'ISLK-V', 'OSIAgr': 'ISLV', 'OSIP-OFDF': 'IStrV-BAZG', 'OSIP-UDSC': 'IStrV-BAZG', 'OSAR': 'ISUV', 'OSIStr': 'ISUV', 'ODiv': 'IvDV', 'ODIV': 'IvDV', 'LAI': 'IVG', 'RAI': 'IVV', 'OAI': 'IVV', 'OSIAC': 'IVZV', 'O OFSPO J+S': 'J+S-V-BASPO', 'O-G+S-UFSPO': 'J+S-V-BASPO', 'LPMFJ': 'JSFVG', 'LPMFV': 'JSFVG', 'OPMFJ': 'JSFVV', 'OPMFV': 'JSFVV', 'LChP': 'JSG', 'LCP': 'JSG', 'DPMin': 'JStG', 'PPMin': 'JStPO', 'OChP': 'JSV', 'OCP': 'VKL', 'OFPC-FINMA': 'KAKV-FINMA', 'OFICol-FINMA': 'KAKV-FINMA', 'OAAcc': 'KBFHV', 'OACust': 'KBFHV', 'LENu': 'KEG', 'OENu': 'KEV', 'LEC': 'KFG', 'LPCu': 'KFG', 'OAVAuto': 'KFZV', 'OAuto': 'KFZV', 'LTBC': 'KGTG', 'OTBC': 'KGTV', 'OIBC': 'KGVV', 'OEBC': 'KGVV', 'LRCN': 'KHG', 'ORCN': 'KHV', 'LIC': 'KIG', 'LEEJ': 'KJFG', 'LPAG': 'KJFG', 'OEEJ': 'KJFV', 'OPAG': 'KJFV', 'LCC': 'KKG', 'OPCC': 'KKV', 'OICol': 'KKV', 'OPC-FINMA': 'KKV-FINMA', 'OICol-FINMA': 'KKV-FINMA', 'OClin': 'KlinV', 'OSRUm': 'KlinV', 'OClin-Dim': 'KlinV-Mep', 'OSRUm-Dmed': 'KlinV-Mep', 'OCI': 'KlV', 'OOCli': 'KlV', 'LFMG': 'KMG', 'LMB': 'KMG', 'OMG': 'KMV', 'OMB': 'KMV', 'OAOF': 'KOV', 'RUF': 'KOV', 'OCoo': 'KoVo', 'OCCRT': 'KoVo', 'OAMédcophy': 'KPAV', 'OMCF': 'KPAV', 'OCPF': 'KPFV', 'FP-TFB': 'KR-PatGer', 'SpG-TFB': 'KR-PatGer', 'OCré-FINMA': 'KreV-FINMA', 'OCre-FINMA': 'KreV-FINMA', 'OEMO': 'KRV', 'ORMT': 'KRV', 'Cst-GE': 'KV-GE', 'Cost.-GE': 'KV-GE', 'LSAMal': 'KVAG', 'LVAMal': 'KVAG', 'LAMal': 'KVG', 'OAMal': 'KVV', 'OPVA': 'LAfV', 'OPSAgr': 'LAfV', 'LRA': 'LBG', 'OAgrD': 'LDV', 'ODAgr': 'LDV', 'OLEl': 'LeV', 'LA': 'LFG', 'LNA': 'LFG', 'OSAv': 'LFV', 'ONA': 'LFV', 'OIBL': 'LGBV', 'OULiq': 'LGBV', 'OGN': 'LGeoIV', 'ODAlOUs': 'LGV', 'ODerr': 'LGV', 'OLiq': 'LiqV', 'OIDAl': 'LIV', 'OID': 'LIV', 'LDAl': 'LMG', 'LDerr': 'LMG', 'OIMLo': 'LMmV', 'OSML': 'LMmV', 'OELDAl': 'LMVV', 'OELDerr': 'LMVV', 'LBFA': 'LPG', 'LAAgr': 'LPG', 'OLRO-FINMA': 'LROV-FINMA', 'OPair': 'LRV', 'OIAt': 'LRV', 'LPPM': 'LSMG', 'OPPM': 'LSMV', 'OPB': 'LSV', 'OIF': 'LSV', 'OTrA': 'LTrV', 'CL': 'LugÜ', 'CLug': 'LugÜ', 'LAP': 'LVG', 'OMN': 'LVV', 'OMN-DDPS': 'LVV-VBS', 'LAgr': 'LwG', 'OAcCP': 'MAkkV', 'OAGio': 'MAkkV', 'OBMa': 'MaLV', 'ORMAp': 'MaLV', 'OMar-FINMA': 'MarV-FINMA', 'OMer-FINMA': 'MarV-FINMA', 'OMach': 'MaschV', 'OMacch': 'MaschV', 'OMat': 'MatV', 'OMAT-DDPS': 'MatV', 'ORM': 'MAV', 'OPart': 'MBV', 'OParC': 'MBV', 'OCMil': 'MCAV', 'OCDM': 'MCAV', 'ODqua': 'MeAV', 'OIQ': 'MeAV', 'ODqua-DFJP': 'MeAV-EJPD', 'OIQ-DFGP': 'MeAV-EJPD', 'LPMéd': 'MedBG', 'LPMed': 'MedBG', 'OPMéd': 'MedBV', 'OPMed': 'MedBV', 'ODim': 'MepV', 'ODmed': 'MepV', 'OSRM': 'MeQV', 'LMétr': 'MessG', 'LMetr': 'MessG', 'OIMes': 'MessMV', 'OStrM': 'MessMV', 'LMét': 'MetG', 'LMet': 'MetG', 'OMét': 'MetV', 'OMet': 'MetV', 'OSV': 'MFV', 'OSVM': 'MFV', 'LAAM': 'MG', 'LM': 'MG', 'OBCBA': 'MGwV', 'OURD': 'MGwV', 'LSIA': 'MIG', 'LSIM': 'MIG', 'OIMin': 'MindStV', 'OImM': 'MindStV', 'OMinTA': 'MinLV', 'Limpmin': 'MinöStG', 'LIOm': 'MinöStG', 'Oimpmin': 'MinöStV', 'OIOm': 'MinöStV', 'LUMin': 'MinVG', 'OUMin': 'MinVV', 'OCL': 'MiPV', 'OSIAr': 'MIV', 'OSIM': 'MIV', 'OCM ES': 'MiVo-HF', 'OERic-SSS': 'MiVo-HF', 'OJM': 'MJV', 'O-GM': 'MJV', 'OAMil': 'MLFV', 'OPCNP': 'MNKPV', 'LPM': 'MSchG', 'OPM': 'MSchV', 'CPM': 'MStG', 'PPM': 'MStP', 'OJPM': 'MStV', 'OGPM': 'MStV', 'O sur la monnaie': 'MünzV', 'OMon': 'MünzV', 'LAM': 'MVG', 'OAM': 'MVV', 'LTVA': 'MWSTG', 'LIVA': 'MWSTG', 'OTVA': 'MWSTV', 'OIVA': 'MWSTV', 'LFORTA': 'NAFG', 'LFOSTRA': 'NAFG', 'ONag': 'NagV', 'OANS': 'NARV', 'OPANS': 'NARV', 'LBN': 'NBG', 'LBNS': 'NBibG', 'OBNS': 'NBibV', 'LRens': 'NDG', 'LAIn': 'NDG', 'ORens': 'NDV', 'OAIn': 'NDV', 'OMBT': 'NEV', 'OPBT': 'NEV', 'OPU': 'NFSV', 'OPE': 'PAVO', 'LPN': 'NHG', 'OPN': 'NHV', 'LRNIS': 'NISSG', 'ORNI': 'NISV', 'OIBT': 'NIV', 'OPCN-OCDE': 'NKPV-OECD', 'OPCN-OCSE': 'NKPV-OECD', 'LVA': 'NSAG', 'LUSN': 'NSAG', 'OVA': 'NSAV', 'OUSN': 'NSAV', 'LRN': 'NSG', 'LSN': 'NSG', 'ORN': 'NSV', 'OSN': 'NSV', 'OARF': 'NZV', 'OARF-OFT': 'NZV-BAV', 'OARF-UFT': 'NZV-BAV', 'OHS-LP': 'OAV-SchKG', 'OAV-LEF': 'OAV-SchKG', 'LAO': 'OBG', 'LMD': 'OBG', 'OAO': 'VUB', 'OMD': 'OBV', 'OPub-FINMA': 'OffV-FINMA', 'LAVI': 'OHG', 'LAV': 'OHG', 'OAVI': 'OHV', 'CO': 'OR', 'OCRDP': 'ÖREBKV', 'OCRDPP': 'ÖREBKV', 'OdelO': 'OrFV', 'OTOr': 'OrFV', 'Org-OMP': 'Org-VöB', 'OOAPub': 'Org-VöB', 'Org ChF': 'OV-BK', 'OrgCaF': 'OV-BK', 'Org CF': 'OV-BR', 'OOrg-CF': 'OV-BR', 'Org DFAE': 'OV-EDA', 'OOrg-DFAE': 'OV-EDA', 'Org DFI': 'OV-EDI', 'OOrg-DFI': 'OV-EDI', 'Org DFF': 'OV-EFD', 'Org-DFF': 'OV-EFD', 'Org DFJP': 'OV-EJPD', 'Org-DFGP': 'OV-EJPD', 'Org LRH': 'OV-HFG', 'Org-LRUm': 'OV-HFG', 'Org DETEC': 'OV-UVEK', 'Org-DATEC': 'OV-UVEK', 'Org-DDPS': 'OV-VBS', 'OOrg-DDPS': 'OV-VBS', 'Org DEFR': 'OV-WBF', 'Org-DEFR': 'OV-WBF', 'LCBr': 'PAG', 'LParl': 'ParlG', 'OLPA': 'VABFP', 'Oparl': 'ParlVV', 'LPart': 'PartG', 'LUD': 'PartG', 'OPTP': 'PaRV', 'OPFP': 'PaRV', 'LBI': 'PatG', 'LTFB': 'PatGG', 'OParcs': 'PäV', 'OPar': 'PäV', 'OAMin': 'PAVO', 'OPTA': 'PAVV', 'LTV': 'PBG', 'OPD-EPF': 'PDV-ETH', 'OPDP-PF': 'PDV-ETH', 'LLG': 'PfG', 'LOF': 'PfG', 'OLG': 'PfV', 'OOF': 'PfV', 'OSaVé': 'PGesV', 'OSalV': 'PGesV', 'OSaVé-DEFR-DETEC': 'PGesV-WBF-UVEK', 'OSalV-DEFR-DATEC': 'PGesV-WBF-UVEK', 'ORPGAA': 'PGRELV', 'ORFGAA': 'PGRELV', 'OPha': 'PhaV', 'OFarm': 'PhaV', 'LOP': 'POG', 'LRFP': 'PrHG', 'LRDP': 'PrHG', 'LSPro': 'PrSG', 'OSPro': 'PrSV', 'ORRTP': 'PRTR-V', 'OPRTR': 'PRTR-V', 'OEPI': 'PSAV', 'ODPI': 'PSAV', 'OPPh': 'PSMV', 'OPF': 'VSB', 'OPsy': 'PsyBV', 'OPPsi': 'PsyBV', 'LPsy': 'PsyG', 'LPPsi': 'PsyG', 'LSPr': 'PüG', 'OPers-METAS': 'PV-METAS', 'OPersTF': 'PVBger', 'OPers-PDHH': 'PVFMH', 'OPers-PRA': 'PVFMH', 'OPers-PDHH-DDPS': 'PVFMH-VBS', 'OPers-PRA-DDPS': 'PVFMH-VBS', 'OPersT': 'PVGer', 'OPers-EPF': 'PVO-ETH', 'OPers PF': 'PVO-ETH', 'OPers-ServAS': 'PVO-TVS', 'OPers-SAT': 'PVO-TVS', 'OPers-PPOE': 'PVSPA', 'OPers-POE': 'PVSPA', 'OPers-PPOE-DDPS': 'PVSPA-VBS', 'OPers-POE-DDPS': 'PVSPA-VBS', 'OIS': 'QStV', 'OIFo': 'QStV', 'OQuaDu': 'QuNaV', 'OQuSo': 'QuNaV', 'R-COPA': 'R-UEK', 'LSR': 'RAG', 'OSRev': 'RAV', 'OEPC-FINMA': 'RelV-FINMA', 'OAPC-FINMA': 'RelV-FINMA', 'RCETF': 'ReRBGer', 'ORe-DFI': 'ResV-EDI', 'ORAMal-DFI': 'ResV-EDI', 'LHR': 'RHG', 'LArRa': 'RHG', 'OHR': 'RHV', 'OArRa': 'RHV', 'OSITC': 'RLSV', 'OITC': 'RLV', 'OrX': 'RöV', 'LAT': 'RPG', 'LPT': 'RPG', 'OAT': 'RPV', 'OPT': 'RPV', 'LRTV': 'RTVG', 'ORTV': 'RTVV', 'OR-AVS': 'RV-AHV', 'LOGA': 'RVOG', 'OLOGA': 'RVOV', 'LASEI': 'SAFIG', 'LASPI': 'SAFIG', 'OPTM': 'SAMV', 'OPLM': 'SAMV', 'OAGa': 'SaV', 'OASal': 'SaV', 'LCFF': 'SBBG', 'LFFS': 'SBBG', 'OMAS': 'SBMV', 'OMSC': 'SBMV', 'LP': 'SchKG', 'LEF': 'SchKG', 'LICa': 'SebG', 'LIFT': 'SebG', 'OICa': 'SebV', 'OIFT': 'SebV', 'OFDG': 'SEFV', 'OFDS': 'SEFV', 'OCâbles': 'SeilV', 'OFuni': 'SeilV', 'OASRE': 'SERV-V', 'OARE': 'SERV-V', 'LASRE': 'SERVG', 'LARE': 'SERVG', 'OPAAb': 'SGV', 'OPeM': 'SGV', 'LEIS': 'SIaG', 'LISDC': 'SIRG', 'CRUGBIN': 'BASEVKGNMD', 'ACSRUGBIN': 'BASEVKGNMD', 'ORIn': 'SnAV', 'ORim': 'SnAV', 'OMJ-DFJP': 'SPBV-EJPD', 'OCG-DFGP': 'SPBV-EJPD', 'OSLing': 'SpDV', 'LLC': 'SpG', 'LLing': 'SpG', 'LESp': 'SpoFöG', 'LPSpo': 'SpoFöG', 'OESp': 'SpoFöV', 'OPSpo': 'SpoFöV', 'LExpl': 'SprstG', 'LEspl': 'SprstG', 'OExpl': 'SprstV', 'OEspl': 'SprstV', 'OLang': 'SpV', 'OLing': 'SpV', 'LVP': 'SRVG', 'LPV': 'SRVG', 'LESE': 'SSchG', 'LSSE': 'SSchG', 'OESE': 'SSchV', 'OSSE': 'SSchV', 'OESE-DFI': 'SSchV-EDI', 'OSSE-DFI': 'SSchV-EDI', 'LNM': 'SSG', 'OSR': 'SSV', 'OSStr': 'SSV', 'LECF': 'StADG', 'LOA': 'StAG', 'LImA': 'StAG', 'LAAF': 'StAhiG', 'OAAF': 'StAhiV', 'OSOA': 'StAV', 'OImA': 'StAV', 'LOAP': 'StBOG', 'OASF': 'STEBV', 'LRCS': 'StFG', 'LCel': 'StFG', 'OPAM': 'StFV', 'OPIR': 'StFV', 'CP': 'StGB', 'LHID': 'StHG', 'LAID': 'StHG', 'OIMRI': 'StMmV', 'OSMRI': 'StMmV', 'CPP': 'StPO', 'LCJ': 'StReG', 'LCaGi': 'StReG', 'OCJ': 'StReV', 'OCaGi': 'StReV', 'LApEl': 'StromVG', 'LAEl': 'StromVG', 'OApEl': 'StromVV', 'OAEl': 'StromVV', 'LRaP': 'StSG', 'ORaP': 'StSV', 'LEnTR': 'STUG', 'LPTS': 'STUG', 'OEnTR': 'STUV', 'OTStr': 'STUV', 'LTRA': 'STVG', 'LTS': 'STVG', 'LSu': 'SuG', 'LRPL': 'SVAG', 'LTTP': 'SVAG', 'ORPL': 'SVAV', 'OTTP': 'SVAV', 'LCR': 'SVG', 'LCStr': 'SVG', 'OS LCart': 'SVKG', 'LPTab': 'TabPG', 'OPTab': 'TabPV', 'OETV 1': 'TAFV 1', 'OETV 2': 'TAFV 2', 'OETV 3': 'TAFV 3', 'OMédv': 'TAMV', 'OMvet': 'TAMV', 'OPBD': 'TBDV', 'OPPD': 'TBDV', 'LVPC': 'TEVG', 'LRVC': 'TEVG', 'OTRF': 'TGBV', 'OSSAn': 'TGDV', 'LETC': 'THG', 'LOTC': 'THG', 'OIMTh': 'TMmV', 'OSMT': 'TMmV', 'LTo': 'ToG', 'OTo': 'ToV', 'OFPT': 'TPFV', 'LTro': 'TrG', 'LIF': 'TrG', 'OFPAn': 'TSchAV', 'LPA': 'TSchG', 'LPAn': 'TSchG', 'OPAn': 'TSchV', 'LFE': 'TSG', 'LTab': 'TStG', 'LImT': 'TStG', 'OITab': 'TStV', 'OImT': 'TStV', 'LET': 'TUG', 'LATC': 'TUG', 'OServAS': 'TVSV', 'OSAT': 'TVSV', 'OPPPS': 'TwwV', 'OPPS': 'VMD', 'LACRE': 'UEG', 'LSgrI': 'UEG', 'OOPA': 'UEV', 'O-COPA': 'UEV', 'Règlement sur la pêche dans le lac Inférieur': 'Unterseefischereiordnung', 'Ordinamento della pesca nel Lago Inferiore': 'Unterseefischereiordnung', 'LTSM': 'UGüTG', 'LTMS': 'UGüTG', 'LIDE': 'UIDG', 'LIDI': 'UIDG', 'OIDE': 'UIDV', 'OIDI': 'UIDV', 'LPtra': 'ÜLG', 'LPTD': 'ÜLG', 'OPtra': 'ÜLV', 'OPTD': 'ÜLV', 'OUMR': 'UraM', 'MMRa': 'UraM', 'LDA': 'URG', 'ODAu': 'URV', 'LPE': 'USG', 'LPAmb': 'USG', 'LAA': 'UVG', 'LAINF': 'UVG', 'OEIE': 'UVPV', 'OEIA': 'UVPV', 'OLAA': 'UVV', 'OAINF': 'UVV', 'LCD': 'UWG', 'LCSl': 'UWG', 'OPersMil': 'V Mil Pers', 'OPers mil': 'V Mil Pers', 'OSEtr': 'V-ASG', 'OSEst': 'V-ASG', 'O-LERI': 'V-FIFG', 'O-LPRI': 'V-FIFG', 'O-LERI-DEFR': 'V-FIFG-WBF', 'O-LPRI-DEFR': 'V-FIFG-WBF', 'OLEH': 'V-GSG', 'OSOsp': 'V-GSG', 'O-LEHE': 'V-HFKG', 'O-LPSU': 'V-HFKG', 'O-STAC': 'V-LTDB', 'O-SIEs': 'V-NDA', 'O-LRNIS': 'V-NISSG', 'O-CNC-FPr': 'V-NQR-BB', 'O QNQ FP': 'V-NQR-BB', 'OCommcon-EPF': 'V-Schliko-ETH', 'O CommConc PF': 'V-Schliko-ETH', 'O-AQIS': 'V-SQWI', 'O-GQIS': 'V-SQWI', 'O-CP-CPM': 'V-StGB-MSt', 'OCP-CPM': 'V-StGB-MSt', 'OPNA': 'VABFP', 'OCDoc': 'VABK', 'RCDoc': 'VABK', 'OETHand': 'VAböV', 'ORTDis': 'VAböV', 'OISofCA': 'VABUA', 'OFSPE': 'VABUA', 'OCFH': 'VAEW', 'OIFI': 'VAEW', 'LSA': 'VAG', 'OMHSI': 'VAI', 'OMFSI': 'VAI', 'OIFC': 'VAK', 'OCFQE': 'VAK', 'OMéd': 'VAM', 'OM': 'VAM', 'OIMEC': 'VAMF', 'OMGC': 'VAMF', 'OMSVM': 'VAmFD', 'OIGE': 'VAMV', 'OSGS': 'VAMV', 'Odac': 'VAN', 'OEAC': 'VAN', 'OSRens': 'VAND', 'OVAIn': 'VAND', 'OLPS': 'VAPF', 'OQPN': 'VAPK', 'OEPIN': 'VAPK', 'OTAS': 'VASA', 'OTaRSi': 'VASA', 'OMBat': 'VASm', 'ONCR': 'VASR', 'OITEP': 'VATPE', 'OITIP': 'VATPE', 'OAHST': 'VATT', 'OATFS': 'VATT', 'OAAFM': 'VATV', 'OASAM': 'VATV', 'OAAFM-DDPS': 'VATV-VBS', 'OASAM-DDPS': 'VATV-VBS', 'ODPO': 'VAU', 'OAPO': 'VAU', 'OInstr pré': 'VAusb', 'OISP': 'VAusb', 'OInstr prém DDPS': 'VAusb-VBS', 'OISP-DDPS': 'VAusb-VBS', 'OMO': 'VAV', 'OMU': 'VAV', 'OMO-DDPS': 'VAV-VBS', 'OMU-DDPS': 'VAV-VBS', 'OLDI': 'VAwG', 'ODI': 'VID', 'OASMéd': 'VAZV', 'OOSM': 'VAZV', 'ODFR': 'VBB', 'Osol': 'VBBo', 'O suolo': 'VBBo', 'OLAr': 'VBGA', 'OLFP': 'VBGF', 'OTrans': 'VBGÖ', 'OTras': 'VBGÖ', 'OIFP': 'VBLN', 'OTUIC': 'VBNIB', 'ODO': 'VBO', 'OOC-SCPT': 'VBO-ÜPF', 'OThand': 'VböV', 'OTDis': 'VböV', 'OPBio': 'VBP', 'OIOP': 'VBPO', 'OOCOP': 'VBPO', 'O-OPers': 'VBPV', 'O-OPers-DFAE': 'VBPV-EDA', 'ORE I': 'VBR I', 'ONE I': 'VBR I', 'ORCSN': 'VBRK', 'ORTN': 'VBRK', 'OEMFP': 'VBSTB', 'OSMFP': 'VBSTB', 'OPSEnt': 'VBSV', 'OPSAz': 'VBSV', 'OAdma': 'VBVA', 'OAE-AF': 'VBVA', 'OGPCT': 'VBVV', 'OABCT': 'VBVV', 'OESN': 'VBWK', 'OCGIN': 'VBWK', 'OCITES': 'VCITES', 'O-CITES': 'VCITES', 'OME-SCPT': 'VD-ÜPF', 'OE-SCPT': 'VD-ÜPF', 'OODA': 'VDA', 'OODE': 'VDA', 'OTPSP': 'VDPS', 'ODPSP': 'VDPS', 'OEDRP-DFI': 'VDPV-EDI', 'OSDRP-DFI': 'VDPV-EDI', 'OCPD': 'VDSZ', 'OTNI': 'VDTI', 'OTDI': 'VDTI', 'OACA': 'VDZV', 'ODCA': 'VDZV', 'OLQE': 'VEAF', 'OLAE': 'VEAF', 'OIELFP': 'VEAGOG', 'OIEVFF': 'VEAGOG', 'OEFMP': 'VEBMP', 'OEE-VVT': 'VEE-PLS', 'OEE-AAT': 'VEE-PLS', 'ODF': 'VEJ', 'OBAF': 'VEJ', 'OGE': 'VEKF', 'OCGE': 'VEKF', 'OEmiA': 'VEL', 'OVotE': 'VEleS', 'OVE': 'VEleS', 'OMETr': 'VEMZ', 'OSTAC': 'VEMZ', 'OESS': 'VES', 'OIIS': 'VES', 'OCREPF': 'VETHBK', 'OCRPF': 'VETHBK', 'OCEl-PA': 'VeÜ-VwV', 'OCE-PA': 'VeÜ-VwV', 'OCEl-PCPP': 'VeÜ-ZSSV', 'OCE-PCPE': 'VeÜ-ZSSV', 'OEV': 'VEV', 'OMoD': 'VeVA', 'OTRif': 'VeVA', 'O E-VERA': 'VEVERA', 'Ordinanza E-VERA': 'VEVERA', 'OIMA': 'VLIb', 'OIMeA': 'VFAI', 'OAFA': 'VFAL', 'OOIT': 'VFAV', 'OPer-Fu': 'VFB-B', 'OLAFum': 'VFB-B', 'OPer-D': 'VFB-DB', 'OADAP': 'VFB-DB', 'OPer-H': 'VFB-G', 'OAS-O': 'VFB-G', 'OPer-B': 'VFB-H', 'OASL': 'VFB-H', 'OPer-Fl': 'VFB-K', 'OASPR': 'VFB-K', 'OPer-AH': 'VFB-LG', 'OASAOG': 'VFB-LG', 'OPer-P': 'VFB-S', 'OALPar': 'VFB-S', 'OPer-S': 'VFB-SB', 'OASSP': 'VFB-SB', 'OPer-Fo': 'VFB-W', 'OASEF': 'VFB-W', 'OVCC': 'VFBF', 'OMA': 'VFD', 'OILAE': 'VFlaW', 'OLDDA': 'VFlaW', 'OLCP': 'VFP', 'Oform': 'VFRR', 'Rform': 'VFRR', 'OSNA': 'VFSD', 'OSA': 'VFSD', 'OAF': 'VFV', 'LRCF': 'VG', 'LResp': 'VG', 'OFCoop': 'VGeK', 'RFCoop': 'VGeK', 'LTAF': 'VGG', 'FITAF': 'VGKE', 'TS-TAF': 'VGKE', 'RTAF': 'VGR', 'OJAr': 'VGS', 'OGD': 'VGS', 'OSPEX': 'VGSEB', 'OASAE': 'VGSEB', 'OEB': 'VGV', 'OIB': 'VGV', 'ODAlGM': 'VGVL', 'ODerrGM': 'VGVL', 'ORegBL': 'VGWR', 'OREA': 'VREG', 'OGOC': 'VHBT', 'OGOCC': 'VHBT', 'OCont': 'VHK', 'OHyPL': 'VHyMP', 'OIgPL': 'VHyMP', 'OHyPPr': 'VHyPrP', 'OIPPrim': 'VHyPrP', 'OHyAb': 'VHyS', 'OIgM': 'VHyS', 'ODIn': 'VID', 'OSIA': 'VIL', 'OILC': 'VILB', 'OSIC': 'VIM', 'OICoM': 'VIM', 'OCMI': 'VIMK', 'OIE': 'VIntA', 'OIntS': 'VIntA', 'OPPEtr': 'VIPaV', 'OIPPE': 'VIPaV', 'OSIS-SRC': 'VIS-NDB', 'OSIME-SIC': 'VIS-NDB', 'OISOS': 'VISOS', 'OVIS': 'VISV', 'OITPTh': 'VITH', 'OITAT': 'VITH', 'OIVS': 'VIVS', 'OCISF': 'ViZG', 'OCISC': 'ViZG', 'OCMIF': 'VIZMB', 'OJAR-FSTD': 'VJAR-FSTD', 'OACata': 'VKA', 'OLCC': 'VKKG', 'OCCEA': 'VKKL', 'OCoC': 'VKKL', 'OSPBC': 'VKKP', 'OSBC': 'VKKP', 'OCPre': 'VKL', 'OCSN': 'VKNS', 'OCos': 'VKos', 'OCTSE': 'VKOVA', 'OPFr': 'VKP', 'OCPPME': 'VKP-KMU', 'OCPPMI': 'VKP-KMU', 'OSSC': 'VKSD', 'Ocach': 'VKSWk', 'OCRCI': 'VKSWk', 'OFA-FINMA': 'VKV-FINMA', 'OCTrI': 'VKVöV', 'OCTrIn': 'VKVöV', 'OMDA': 'VKZ', 'OCA': 'VVK', 'OBNP': 'VLBE', 'ODPPE': 'VLBE', 'OBCF': 'VLE', 'ORFF': 'VLE', 'ORAgr': 'VLF', 'LCo': 'VlG', 'OECA': 'VLHb', 'OICA': 'VLHb', 'OOMA': 'VLIb', 'OPEA': 'VLIp', 'OCDA': 'VLKA', 'OCDAE': 'VLKA', 'ONAE': 'VLL', 'ODNA': 'VLL', 'ODAlOV': 'VLpH', 'ODOV': 'VLpH', 'ODAlAn': 'VLtH', 'ODOA': 'VLtH', 'OCo': 'VlV', 'OEMTP': 'VMAP', 'OSMLP': 'VMAP', 'OAMAS': 'VMBM', 'OAMM': 'VMBM', 'ODPS': 'VMD', 'OMi': 'VMDP', 'OOPSM': 'VMDP', 'OACAMIL': 'VMILAK', 'OACMIL': 'VMILAK', 'OAMC': 'VmKI', 'OMob': 'VMob', 'OSM': 'VMS', 'ONM': 'VMSch', 'OCM': 'VMSV', 'OCSM': 'VMSV', 'OBLF': 'VMWG', 'OLAL': 'VMWG', 'OOPT': 'VNEF', 'OORT': 'VNEF', 'OCNE': 'VNEK', 'OCAl': 'VNem', 'OIAl': 'VNem', 'OUS': 'VNF', 'OAPub': 'VöB', 'OCOV': 'VOCV', 'OSO': 'VOD', 'OOBE': 'VOEW', 'OOSE': 'VOEW', 'OOSG': 'VOGW', 'OCoR': 'VORA', 'OCoR-DFI': 'VORA-EDI', 'OTN': 'VOSA', 'OPoA': 'VPA', 'OPPE': 'VPA', 'ORCPP': 'VPABP', 'OPPCPers': 'VPABP', 'OSAss': 'VPAV', 'RPAss': 'VPAV', 'OTV': 'VPB', 'OPIE': 'VPeA', 'OPAR': 'VPEV', 'ORPAR': 'VPEV', 'OAPA': 'VPGA', 'ORInt': 'VPiB', 'OMP-OFEV': 'VpM-BAFU', 'OMF-UFAM': 'VpM-BAFU', 'OMP-OFAG': 'VpM-BLW', 'OMF-UFAG': 'VpM-BLW', 'OOP EPF': 'VPO ETH', 'OOP-PF': 'VPO ETH', 'OOPC': 'VPOB', 'OFipo': 'VPofi', 'OFiPo': 'VPofi', 'OLOP': 'VPOG', 'OOP': 'VPOG', 'ODP': 'VPR', 'OMAP': 'VPRG', 'ODFCLSI': 'VPRG', 'OPOVA': 'VPRH', 'OAOVA': 'VPRH', 'OPPr': 'VPrP', 'OPPrim': 'VPrP', 'OPSP': 'VPS', 'OCSP': 'VPSP', 'OEnB': 'VPV', 'RPB': 'VPV', 'OPAPIF': 'VPVE', 'ORPM': 'VPVK', 'ORPMUE': 'VPVKEU', 'RP-EPF 1': 'VR-ETH 1', 'RP-PF 1': 'VR-ETH 1', 'RP-EPF 2': 'VR-ETH 2', 'RP-PF 2': 'VR-ETH 2', 'OCBD': 'VRA', 'OCQO': 'VRA', 'RPEC': 'VRAB', 'RPIC': 'VRAB', 'ORSAE': 'VREG', 'OSCR': 'VRKD', 'ORésDAlan': 'VRLtH', 'ORDOA': 'VRLtH', 'OPR': 'VRP', 'ORCPL': 'VRSL', 'OReq': 'VRSL', 'ORA': 'VRV-L', 'ONCA': 'VRV-L', 'OPCF': 'VSB', 'OEMCN': 'VSBN', 'OSMCN': 'VSBN', 'OAbCV': 'VSFK', 'ODCS': 'VSFS', 'ODSRS': 'VSFS', 'OOCCR-OFROU': 'VSKV-ASTRA', 'OOCCS-USTR': 'VSKV-ASTRA', 'OMSA': 'VSL', 'OSMP': 'VSMS', 'OMSM': 'VSMS', 'ODiTr': 'VSoTr', 'ODiT': 'VSoTr', 'OPPBE': 'VSPA', 'OPBE': 'VSPA', 'OPESp': 'VSpoFöP', 'OPPSpo': 'VSpoFöP', 'OPPB': 'VSPS', 'ORS': 'VSR', 'ORSA': 'VSRL', 'OSJo': 'VSS', 'OSG': 'VSS', 'OOST': 'VST', 'OOSI': 'VST', 'ORCS': 'VStFG', 'ORCel': 'VStFG', 'LIA': 'VStG', 'LIP': 'VStG', 'DPA': 'VStrR', 'OIA': 'VStV', 'OIPrev': 'VStV', 'OSAA': 'VSUV', 'OSAI': 'VSUV', 'OFDPP': 'VSVB', 'OFDP': 'VSVB', 'OEIT': 'VSZV', 'OIET': 'VSZV', 'OCVM': 'VTE', 'OVF': 'VTE', 'OAP': 'VTM', 'OAAP': 'VTM', 'OSPA': 'VTNP', 'OSOAn': 'VTNP', 'OETV': 'VTS', 'OPAnAb': 'VTSchS', 'OPAnMac': 'VTSchS', 'OPAT': 'VWS', 'OPrTec': 'VtVtH', 'OOr': 'VUB', 'OOr-DEFR': 'VUB-WBF', 'OAO-DEFR': 'VUB-WBF', 'OFSPers': 'VUFB', 'OCPCP': 'VUKBK', 'OCIPP': 'VUKBK', 'OACM': 'VUM', 'OAAM': 'VUM', 'OSCPT': 'VÜPF', 'OPA': 'VUV', 'OPI': 'VUV', 'OVid-TP': 'VüV-ÖV', 'OVsor-TP': 'VüV-ÖV', 'OROPD': 'VUZPE', 'OROPS': 'VUZPE', 'OAA': 'VVA', 'OAE': 'VVA', 'OPC': 'VVAG', 'ODiC': 'VVAG', 'OOLDI': 'VVAwG', 'O-ODI-DFAE': 'VVAwG', 'OISec': 'VVE', 'OFIM': 'VVE', 'OLED': 'VVEA', 'OPSR': 'VVEA', 'LCA': 'VVG', 'OTeA': 'VVK', 'OCA-DFI': 'VVK-EDI', 'OTeA-DFI': 'VVK-EDI', 'OMAH': 'VVMH', 'OMPAH': 'VVMH', 'OOUS': 'VVNF', 'OUUS': 'VVNF', 'OST-SCPT': 'VVS-ÜPF', 'OPSE': 'VVSG', 'OPreS': 'VVSG', 'OERE': 'VVWAL', 'OEAE': 'VVWAL', 'LALM': 'VWBG', 'LMAM': 'VWBG', 'OLCAP': 'VWEG', 'OFSI': 'VWEV', 'OCMSD': 'VWEV', 'OSS': 'VWL', 'OAEP': 'VWLV', 'OPATE': 'VWS', 'PA': 'VwVG', 'OASA': 'VZAE', 'LGEL': 'VZEG', 'LPIL': 'VZEG', 'OSCSE': 'VZertES', 'OFiEle': 'VZertES', 'ORFI': 'VZG', 'RFF': 'VZG', 'ONCAF': 'VZSchB', 'OAC': 'VZV', 'OASM': 'VZVM', 'OAVM': 'VZVM', 'LFo': 'WaG', 'OFo': 'WaV', 'LFCo': 'WeBiG', 'OFCo': 'WeBiV', 'OEPL': 'WEFV', 'OPPA': 'WEFV', 'LCAP': 'WEG', 'LArm': 'WG', 'LAMO': 'WHG', 'LTEO': 'WPEG', 'OTEO': 'WPEV', 'OIRH': 'WResV', 'OREI': 'WResV', 'LFH': 'WRG', 'LUFI': 'WRG', 'OFH': 'WRV', 'OUFI': 'WRV', 'LPAP': 'WSchG', 'LPSt': 'WSchG', 'OPAP': 'WSchV', 'OPSt': 'WSchV', 'OMT': 'WTO', 'OArm': 'WV', 'LUMMP': 'WZG', 'LUMP': 'WZG', 'RDE': 'WZV', 'ODA': 'WZV', 'OROEM': 'WZVV', 'ORUAM': 'WZVV', 'LUsC': 'ZAG', 'LCoe': 'ZAG', 'LFisE': 'ZBstG', 'LFR': 'ZBstG', 'LSC': 'ZDG', 'ODSC': 'ZDUeV', 'OSCi': 'ZDV', 'OSCi-DEFR': 'ZDV-WBF', 'LDIF': 'ZEBG', 'LSIF': 'ZEBG', 'LOC': 'ZentG', 'LUC': 'ZentG', 'SCSE': 'ZertES', 'FiEle': 'ZertES', 'Ltém': 'ZeugSG', 'LPTes': 'ZeugSG', 'OTém': 'ZeugSV', 'OPTes': 'ZeugSV', 'OADou': 'ZEV', 'OADo': 'ZEV', 'LD': 'ZG', 'CC': 'ZGB', 'LCPI': 'ZISG', 'OCMétr': 'ZMessV', 'OCMetr': 'ZMessV', 'CPC': 'ZPO', 'CCoop-ESF': 'ZSAV-BiZ', 'CColl-SFS': 'ZSAV-BiZ', 'CCoop-HE': 'ZSAV-HS', 'ConSU': 'ZSAV-HS', 'OAASF': 'ZSTEBV', 'OEEC': 'ZStGV', 'OESC': 'ZStGV', 'OEC': 'ZStV', 'OSC': 'ZStV', 'OPCi': 'ZSV', 'LTaD': 'ZTG', 'LTD': 'ZTG', 'LAS': 'ZUG', 'OComp-OSPro': 'ZustV-PrSV', 'OAdd': 'ZuV', 'OD': 'ZV', 'OD-OFDF': 'ZV-BAZG', 'OD-UDSC': 'ZV-BAZG', 'OD-DFF': 'ZV-EFD', 'OA-DFJP': 'ZV-EJPD', 'OA-DFGP': 'ZV-EJPD', 'LRS': 'ZWG', 'LASec': 'ZWG', 'ORSec': 'ZWV', 'OASec': 'ZWV'}
print(f'multilang entries: {len(MULTILANG_ABBR)}')

In [ ]:
# Cell 3 — citation parsing, normalization, corpus index, granularity filter
ART_PATTERN = re.compile(
    r'^Art\.\s*(?P<art>\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*Abs\.\s*(?P<abs>\d+[a-z]?(?:bis|ter)?))?'
    r'(?:\s*lit\.\s*(?P<lit>[a-zA-Z]+))?'
    r'(?:\s*Ziff\.\s*(?P<ziff>\d+))?'
    r'\s+(?P<code>[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)$'
)

def parse_article(c):
    m = ART_PATTERN.match(c.strip()); return m.groupdict() if m else None

def normalize_whitespace(c):
    s = re.sub(r'\s+', ' ', c.strip())
    s = re.sub(r'\bArt\.(?=\d)', 'Art. ', s)
    s = re.sub(r'\bAbs\.(?=\d)', 'Abs. ', s)
    s = re.sub(r'\blit\.(?=[a-zA-Z])', 'lit. ', s)
    s = re.sub(r'\bZiff\.(?=\d)', 'Ziff. ', s)
    return s

def expand_multilang(c):
    out = [c]
    m = parse_article(c)
    if m and m['code'] in MULTILANG_ABBR:
        code_de = MULTILANG_ABBR[m['code']]
        out.append(c[:c.rfind(m['code'])] + code_de)
    return out

print('Loading laws & court citation sets ...')
laws_df = pd.read_csv(DATA / 'laws_de.csv')
laws_cits = set(laws_df['citation'].astype(str))
court_cits = set()
with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row: court_cits.add(row[0])
CORPUS = laws_cits | court_cits
print(f'corpus citations: laws={len(laws_cits):,}  court={len(court_cits):,}  combined={len(CORPUS):,}')

PARENT_TO_CHILDREN = defaultdict(list)
for c in laws_cits:
    m = ART_PATTERN.match(c)
    if not m: continue
    parent = f"Art. {m.group('art')} {m.group('code')}"
    if c != parent:
        PARENT_TO_CHILDREN[parent].append(c)
print(f'parents with children: {len(PARENT_TO_CHILDREN):,}')

def resolve_against_corpus(cit):
    c = normalize_whitespace(cit)
    if c in CORPUS: return [c]
    for v in expand_multilang(c):
        if v != c and v in CORPUS: return [v]
        if v != c and v in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[v])
    if c in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[c])
    m = parse_article(c)
    if m:
        parent = f"Art. {m['art']} {m['code']}"
        if parent != c and parent in CORPUS: return [parent]
        for v in expand_multilang(parent):
            if v != c and v in CORPUS: return [v]
            if v != c and v in PARENT_TO_CHILDREN: return list(PARENT_TO_CHILDREN[v])
    return []

def granularity_filter(cits):
    cs = set(cits); keep=[]
    for c in cits:
        kids = PARENT_TO_CHILDREN.get(c)
        if kids and any(k in cs for k in kids): continue
        keep.append(c)
    return keep

def dedup(cits):
    seen=set(); out=[]
    for c in cits:
        if c not in seen: seen.add(c); out.append(c)
    return out

# sanity: train resolve rate
train_df = pd.read_csv(DATA / 'train.csv')
def split_cits(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [t.strip() for t in s.split(';') if t.strip()]
total=resolved=0
for _, r in train_df.iterrows():
    for g in split_cits(r['gold_citations']):
        total+=1
        if resolve_against_corpus(g): resolved+=1
print(f'TRAIN gold resolve: {resolved}/{total} = {resolved/max(1,total):.4f}')
assert resolved/max(1,total) > 0.95, 'normalization regressed below 95% — investigate'


In [ ]:
# Cell 4 — BGE-M3 encoder on cuda:0 (auto-discover path, max_seq=512)
import torch, gc, os
from pathlib import Path
from sentence_transformers import SentenceTransformer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# GPU layout (T4 x2):
#   cuda:0 → BGE-M3 (this cell, encoder for laws+court+queries)
#   cuda:1 → NLLB, reranker, Qwen (smaller models, see cells 7/8/11)
ENCODER_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

def discover_bge_m3_paths():
    paths = []
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for root, dirs, files in os.walk(kaggle_input):
            if 'config.json' not in files:
                continue
            root_lower = root.lower()
            if 'bge' in root_lower or '/m3/' in root_lower + '/' or root_lower.endswith('/m3'):
                paths.append(root)
    paths.extend([
        '/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1',
        '/kaggle/input/bge-m3/transformers/bge-m3/1',
        '/kaggle/input/bge-m3',
        'BAAI/bge-m3',
    ])
    seen=set(); out=[]
    for p in paths:
        if p not in seen: seen.add(p); out.append(p)
    return out

print('/kaggle/input/ layout (depth 2):')
ki = Path('/kaggle/input')
if ki.exists():
    for entry in sorted(ki.iterdir()):
        print(f'  {entry.name}/')
        if entry.is_dir():
            try:
                for sub in sorted(entry.iterdir())[:5]:
                    print(f'    {sub.name}/')
            except PermissionError:
                pass

BGE_M3_PATHS = discover_bge_m3_paths()
print(f'\n{len(BGE_M3_PATHS)} candidate paths:')
for p in BGE_M3_PATHS:
    exists = '\u2713' if (p.startswith('/') and Path(p).exists()) else ('hub' if not p.startswith('/') else '\u00d7')
    print(f'  [{exists}] {p}')

ENCODER = None
for p in BGE_M3_PATHS:
    try:
        print(f'\ntrying {p} ...')
        ENCODER = SentenceTransformer(p, device=ENCODER_DEVICE)
        print(f'  \u2705 BGE-M3 loaded from {p} on {ENCODER_DEVICE}')
        break
    except Exception as e:
        print(f'  failed: {type(e).__name__}: {str(e)[:120]}')

if ENCODER is None:
    raise RuntimeError('BGE-M3 not available. Attach model or enable Internet.')

# fp16 + short seq for memory headroom during heavy encoding
if torch.cuda.is_available():
    try:
        ENCODER.half()
    except Exception:
        pass
try:
    ENCODER.max_seq_length = 512
    print(f'  max_seq_length = 512')
except Exception:
    pass

def encode(texts, batch_size=32, show_progress=False):
    return ENCODER.encode(
        texts, batch_size=batch_size, normalize_embeddings=True,
        show_progress_bar=show_progress, convert_to_numpy=True,
    ).astype('float32')

def encode_query(q):
    return encode([q], batch_size=1)

# Report VRAM after load
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i}  free={free/1e9:.1f}GB / total={total/1e9:.1f}GB')


In [ ]:
# Cell 5 — laws_de BM25 + BGE-M3 dense (cuda:0, batch 32, chunked, cached)
import faiss
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r'[A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9]+')
def tokenize(text):
    if not isinstance(text, str): return []
    return [t.lower() for t in TOKEN_RE.findall(text)]

laws_citations = laws_df['citation'].astype(str).tolist()
laws_texts = laws_df['text'].fillna('').astype(str).tolist()

bm25_path = CACHE / 'laws_bm25.pkl'
if bm25_path.exists():
    print('Loading cached laws BM25 ...')
    with open(bm25_path, 'rb') as f: bm25 = pickle.load(f)
else:
    print(f'Tokenizing & building BM25 ...')
    t0 = time.time()
    tokenized = [tokenize(t) for t in laws_texts]
    bm25 = BM25Okapi(tokenized)
    with open(bm25_path, 'wb') as f: pickle.dump(bm25, f)
    print(f'  BM25 built in {time.time()-t0:.1f}s')

faiss_path = CACHE / 'laws_dense.faiss'
if faiss_path.exists():
    print('Loading cached laws FAISS ...')
    laws_dense = faiss.read_index(str(faiss_path))
else:
    CHUNK = 10000
    BATCH = 32
    n = len(laws_texts)
    print(f'Encoding {n:,} laws on {ENCODER_DEVICE}  (chunks={CHUNK}, batch={BATCH}, seq=512)')
    t0 = time.time()
    parts = []
    for start in range(0, n, CHUNK):
        end = min(start + CHUNK, n)
        sub = laws_texts[start:end]
        emb_chunk = ENCODER.encode(
            sub, batch_size=BATCH, normalize_embeddings=True,
            show_progress_bar=False, convert_to_numpy=True,
        ).astype('float32')
        parts.append(emb_chunk)
        elapsed = time.time() - t0
        eta = elapsed / max(1, end) * (n - end)
        print(f'  [{end:>6d}/{n}]  elapsed {elapsed:.1f}s  ETA {eta:.0f}s', flush=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    emb = np.vstack(parts)
    del parts; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f'  done in {time.time()-t0:.1f}s  shape={emb.shape}')

    laws_dense = faiss.IndexFlatIP(emb.shape[1])
    laws_dense.add(emb)
    faiss.write_index(laws_dense, str(faiss_path))
    del emb; gc.collect()

print(f'laws FAISS ready: ntotal={laws_dense.ntotal}  d={laws_dense.d}')


In [ ]:
# Cell 6 — court 2-stage (cuda:0, batch 32, chunked, e_texts persisted)
RE_BGE  = re.compile(r'^BGE\s+\d+\s+[IVX]+\s+\d+')
RE_NEW  = re.compile(r'^\d+[A-Za-z]_\d+/\d+')
RE_LEG  = re.compile(r'^([A-Z])\s*\d+/\d+\s+\d{2}\.\d{2}\.\d{4}')
RE_LEGC = re.compile(r'^\d+[A-Z]\.\d+/\d+\s+\d{2}\.\d{2}\.\d{4}')

def judgment_id(cit):
    for r in (RE_BGE, RE_NEW, RE_LEG, RE_LEGC):
        m = r.match(cit)
        if m: return m.group(0)
    return None

court_meta_path = CACHE / 'court_meta.pkl'
court_jfaiss    = CACHE / 'court_j.faiss'
court_efaiss    = CACHE / 'court_e.faiss'
E_TEXT_TRUNC = 400

if USE_COURT and court_meta_path.exists() and court_jfaiss.exists() and court_efaiss.exists():
    print('Loading cached court indexes ...')
    with open(court_meta_path, 'rb') as f: court_meta = pickle.load(f)
    j_dense = faiss.read_index(str(court_jfaiss))
    e_dense = faiss.read_index(str(court_efaiss))
    if 'e_texts_trunc' not in court_meta:
        print('  cache missing e_texts_trunc — rebuilding ...')
        existing_cits = set(court_meta['e_citations'])
        e_texts_lookup = {}
        with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
            r = csv.reader(f); next(r, None)
            for row in r:
                if not row or len(row) < 2: continue
                cit = row[0]
                if cit in existing_cits:
                    e_texts_lookup[cit] = (row[1] or '')[:E_TEXT_TRUNC]
        court_meta['e_texts_trunc'] = [e_texts_lookup.get(c, '') for c in court_meta['e_citations']]
        with open(court_meta_path, 'wb') as f: pickle.dump(court_meta, f)
        print('  rebuilt e_texts_trunc')
elif USE_COURT:
    print('Streaming court_considerations.csv ...')
    bucket=defaultdict(list); e_cits=[]; e_jids=[]; e_texts=[]
    t0=time.time()
    E_CAP, CHAR_CAP = 30, 4000
    with open(DATA / 'court_considerations.csv', encoding='utf-8') as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if not row or len(row) < 2: continue
            cit, text = row[0], row[1] or ''
            jid = judgment_id(cit)
            if jid is None: continue
            e_cits.append(cit); e_jids.append(jid); e_texts.append(text)
            if len(bucket[jid]) < E_CAP: bucket[jid].append(text)
    print(f'  streamed in {time.time()-t0:.1f}s  E={len(e_cits):,}  judgments={len(bucket):,}')

    judgment_ids = list(bucket.keys())
    j2idx = {j:i for i,j in enumerate(judgment_ids)}
    judgment_texts = ['\n'.join(bucket[j])[:CHAR_CAP] for j in judgment_ids]
    e_to_jidx = [j2idx[j] for j in e_jids]

    def encode_chunked(texts, batch, chunk, label):
        n = len(texts)
        print(f'Encoding {n:,} {label} on {ENCODER_DEVICE}  (chunks={chunk}, batch={batch}, seq=512)')
        t0 = time.time()
        parts = []
        for start in range(0, n, chunk):
            end = min(start + chunk, n)
            sub = texts[start:end]
            emb_chunk = ENCODER.encode(
                sub, batch_size=batch, normalize_embeddings=True,
                show_progress_bar=False, convert_to_numpy=True,
            ).astype('float32')
            parts.append(emb_chunk)
            elapsed = time.time() - t0
            eta = elapsed / max(1, end) * (n - end)
            print(f'  [{end:>7d}/{n}]  elapsed {elapsed:.1f}s  ETA {eta:.0f}s', flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        emb = np.vstack(parts)
        del parts; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'  done in {time.time()-t0:.1f}s  shape={emb.shape}')
        return emb

    j_emb = encode_chunked(judgment_texts, batch=32, chunk=5000, label='judgments')
    j_dense = faiss.IndexFlatIP(j_emb.shape[1]); j_dense.add(j_emb)
    del j_emb; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    e_emb = encode_chunked(e_texts, batch=32, chunk=20000, label='E-level')
    if len(e_emb) > 50_000:
        d = e_emb.shape[1]; nlist=4096
        quant = faiss.IndexFlatIP(d)
        e_dense = faiss.IndexIVFPQ(quant, d, nlist, 64, 8)
        e_dense.train(e_emb); e_dense.add(e_emb); e_dense.nprobe = 32
    else:
        e_dense = faiss.IndexFlatIP(e_emb.shape[1]); e_dense.add(e_emb)
    del e_emb; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    e_texts_trunc = [t[:E_TEXT_TRUNC] for t in e_texts]
    del e_texts; gc.collect()

    court_meta = {
        'judgment_ids':  judgment_ids, 'e_citations': e_cits,
        'e_to_jidx': e_to_jidx, 'e_texts_trunc': e_texts_trunc,
    }
    with open(court_meta_path, 'wb') as f: pickle.dump(court_meta, f)
    faiss.write_index(j_dense, str(court_jfaiss))
    faiss.write_index(e_dense, str(court_efaiss))
else:
    print('USE_COURT=False')
    court_meta = {'judgment_ids':[], 'e_citations':[], 'e_to_jidx':[], 'e_texts_trunc':[]}
    j_dense = None; e_dense = None

if USE_COURT:
    print(f'court ready: judgments={len(court_meta["judgment_ids"]):,}  '
          f'E-rows={len(court_meta["e_citations"]):,}  '
          f'has_text={bool(court_meta.get("e_texts_trunc"))}')


In [ ]:
# Cell 7 — NLLB EN→DE translation on cuda:1 (fail-soft)
SECOND_DEVICE = 'cuda:1' if (torch.cuda.is_available() and torch.cuda.device_count() >= 2) else (
    'cuda:0' if torch.cuda.is_available() else 'cpu')

TRANSLATOR = None
TRANSLATOR_TOK = None
if USE_TRANSLATE:
    NLLB_PATHS = [
        '/kaggle/input/nllb-200-distilled/nllb-200-distilled-600M',
        '/kaggle/input/nllb-200-distilled-600m',
        '/kaggle/input/nllb-200-distilled',
        'facebook/nllb-200-distilled-600M',
    ]
    # also discover from /kaggle/input/models/
    extra = []
    ki = Path('/kaggle/input')
    if ki.exists():
        for root, dirs, files in os.walk(ki):
            if 'config.json' in files and 'nllb' in root.lower():
                extra.append(root)
    NLLB_PATHS = extra + NLLB_PATHS

    for p in NLLB_PATHS:
        try:
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
            TRANSLATOR_TOK = AutoTokenizer.from_pretrained(p, src_lang='eng_Latn')
            TRANSLATOR = AutoModelForSeq2SeqLM.from_pretrained(p, torch_dtype=torch.float16)
            TRANSLATOR = TRANSLATOR.to(SECOND_DEVICE)
            print(f'NLLB loaded on {SECOND_DEVICE} from {p}')
            break
        except Exception as e:
            print(f'  NLLB {p} failed: {type(e).__name__}: {str(e)[:80]}')
    if TRANSLATOR is None:
        print('  NLLB unavailable — fallback identity translate')

def translate_to_de(text):
    if TRANSLATOR is None: return text
    try:
        inputs = TRANSLATOR_TOK(text, return_tensors='pt', truncation=True, max_length=512)
        dev = next(TRANSLATOR.parameters()).device
        inputs = {k: v.to(dev) for k, v in inputs.items()}
        forced = TRANSLATOR_TOK.convert_tokens_to_ids('deu_Latn')
        with torch.no_grad():
            out = TRANSLATOR.generate(**inputs, forced_bos_token_id=forced,
                                       max_new_tokens=384, num_beams=1)
        return TRANSLATOR_TOK.batch_decode(out, skip_special_tokens=True)[0]
    except Exception:
        return text

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i}  free={free/1e9:.1f}GB / total={total/1e9:.1f}GB')


In [ ]:
# Cell 8 — Cross-encoder reranker on cuda:1 (fail-soft)
RERANKER = None
if USE_RERANKER:
    RERANK_PATHS = [
        '/kaggle/input/bge-reranker-v2-m3/bge-reranker-v2-m3',
        '/kaggle/input/bge-reranker-v2-m3',
        '/kaggle/input/baai-bge-reranker-v2-m3',
        'BAAI/bge-reranker-v2-m3',
    ]
    extra = []
    ki = Path('/kaggle/input')
    if ki.exists():
        for root, dirs, files in os.walk(ki):
            if 'config.json' in files and 'rerank' in root.lower():
                extra.append(root)
    RERANK_PATHS = extra + RERANK_PATHS

    from sentence_transformers import CrossEncoder
    for p in RERANK_PATHS:
        try:
            RERANKER = CrossEncoder(p, device=SECOND_DEVICE)
            print(f'Reranker loaded on {SECOND_DEVICE} from {p}')
            break
        except Exception as e:
            print(f'  reranker {p} failed: {type(e).__name__}: {str(e)[:80]}')
    if RERANKER is None:
        print('  Reranker unavailable')

def cross_rerank(query, pairs, top_k=100):
    if RERANKER is None or not pairs:
        return [(c, 0.0) for c, _ in pairs[:top_k]]
    try:
        scores = RERANKER.predict([[query, t or ''] for _, t in pairs],
                                   batch_size=32, show_progress_bar=False)
    except TypeError:
        scores = RERANKER.predict([[query, t or ''] for _, t in pairs])
    scored = list(zip([c for c,_ in pairs], (float(s) for s in scores)))
    scored.sort(key=lambda kv: -kv[1])
    return scored[:top_k]

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i}  free={free/1e9:.1f}GB / total={total/1e9:.1f}GB')


In [ ]:
# Cell 9 — universal defaults + helpers (topic anchors NOT force-injected this time)
# Lesson from phase 6: hand-crafted topic anchors overfit val & crowded out signal.
# Here we keep only the universal default (Art. 100 Abs. 1 BGG: 9/10 val) and let the
# LLM generate the rest. classify_topic kept only for optional light prior.

UNIVERSAL_DEFAULTS = ['Art. 100 Abs. 1 BGG']

def get_universal():
    out = []
    for d in UNIVERSAL_DEFAULTS:
        r = resolve_against_corpus(d)
        if r: out.append(r[0])
    return dedup(out)

# Citation-form helpers for quality control
import re as _re_q
_SR_NUMERIC_RE = _re_q.compile(r'\s(\d{3}(?:\.\d+)+(?:[a-z])?)\s*$')
def code_of(cit):
    parts = cit.rsplit(' ', 1)
    return parts[-1] if parts else cit
def is_sr_numeric(cit):
    return bool(_SR_NUMERIC_RE.search(cit))


In [ ]:
# Cell 10 — Deterministic signals from train + EDA observations

# === 10.1 Train citation co-occurrence ===
from collections import Counter, defaultdict

train_df = pd.read_csv(DATA / 'train.csv')
COOCCUR = defaultdict(Counter)
CIT_FREQ = Counter()
for s in train_df['gold_citations'].fillna(''):
    cits = [c.strip() for c in s.split(';') if c.strip()]
    for c in cits: CIT_FREQ[c] += 1
    for i, a in enumerate(cits):
        for b in cits[i+1:]:
            if a != b:
                COOCCUR[a][b] += 1
                COOCCUR[b][a] += 1
print(f'train co-occur: {len(CIT_FREQ):,} cits, '
      f'{sum(len(v) for v in COOCCUR.values())//2:,} pairs')

def cooccur_expand(seed_cits, top_k=5, min_count=2):
    out = []
    for s in seed_cits:
        if s not in COOCCUR: continue
        for cand, cnt in COOCCUR[s].most_common(top_k):
            if cnt >= min_count and cand not in out and cand != s:
                out.append(cand)
    return out

# === 10.2 Code frequency prior ===
CODE_FREQ = Counter()
for c in CIT_FREQ:
    parts = c.rsplit(' ', 1)
    if len(parts) == 2:
        CODE_FREQ[parts[-1]] += CIT_FREQ[c]
_max = max(CODE_FREQ.values()) if CODE_FREQ else 1
CODE_BOOST = {code: 0.5 + (cnt / _max) for code, cnt in CODE_FREQ.items()}
print(f'code top 10: {CODE_FREQ.most_common(10)}')

def code_boost_of(cit):
    parts = cit.rsplit(' ', 1)
    if len(parts) != 2: return 0.5
    return CODE_BOOST.get(parts[-1], 0.5)

# === 10.3 Same-judgment E. expansion ===
RE_E_SUFFIX = re.compile(r'(.*?)\s+E\.\s*([\d\.]+)\s*$')

def judgment_root(cit):
    m = RE_E_SUFFIX.match(cit)
    if m: return m.group(1), m.group(2)
    return cit, None

JUDGMENT_TO_ES = defaultdict(list)
if USE_COURT and court_meta.get('e_citations'):
    for cit in court_meta['e_citations']:
        root, e = judgment_root(cit)
        if e is not None:
            JUDGMENT_TO_ES[root].append(cit)
print(f'judgment→Es index: {len(JUDGMENT_TO_ES):,} judgments')

def same_judgment_expand(court_hits, max_per_judgment=4):
    seen_judgments = set()
    out = []
    for cit in court_hits:
        root, e = judgment_root(cit)
        if e is None or root in seen_judgments: continue
        seen_judgments.add(root)
        siblings = JUDGMENT_TO_ES.get(root, [])
        for sib in siblings[:max_per_judgment]:
            if sib != cit and sib not in out:
                out.append(sib)
    return out

# === 10.4 Lexical-mention strength ===
RE_LIT_CITE = re.compile(
    r'\b(?:Art(?:icle)?\.?\s*\d+|BGE\s+\d+|\d+[A-Z]_\d+/\d+)',
    re.IGNORECASE,
)
def lexical_mention_count(query):
    return len(RE_LIT_CITE.findall(query))
def lexical_strength(query):
    n = lexical_mention_count(query)
    if n >= 5: return 'high'
    if n >= 2: return 'mid'
    return 'low'

# === 10.5 Form distribution ===
def predict_form_mix(query):
    ql = query.lower()
    bge = ['bge', 'leading case', 'supreme court']
    bger = ['bger', 'federal supreme court ruling', 'cantonal court']
    art_only = ['article ', 'art. ', 'pursuant to ', 'in accordance with']
    has_bge = any(s in ql for s in bge)
    has_bger = any(s in ql for s in bger)
    has_art = any(s in ql for s in art_only)
    if has_bge or has_bger: return 0.6, 1.0
    if has_art and not (has_bge or has_bger): return 1.0, 0.6
    return 1.0, 1.0


In [ ]:
# Cell 11 — European LLM loader (auto-discover, 4-bit, fail-soft)
LLM = None; LLM_TOK = None; LLM_FAILED = False
USE_LLM_4BIT = True

# Priority: bigger/European first. Auto-discovered from /kaggle/input/ + explicit fallbacks.
def discover_llm_paths():
    import os
    paths = []
    ki = Path('/kaggle/input')
    # priority: bigger/European/German-legal first
    KEYS = ['mistral-small', 'mixtral', 'em-german', 'em_german', 'leo-mistral',
            'leolm', 'mistral-nemo', 'mistral-ne', 'eurollm', 'teuken',
            'mistral', 'llama-3', 'qwen']
    found = {k: [] for k in KEYS}
    if ki.exists():
        for root, dirs, files in os.walk(ki):
            if 'config.json' not in files:
                continue
            rl = root.lower()
            for k in KEYS:
                if k in rl:
                    found[k].append(root)
                    break
    for k in KEYS:
        paths.extend(sorted(found[k]))
    # explicit hub fallbacks (Internet ON)
    paths.extend([
        'mistralai/Mistral-Small-Instruct-2409',
        'jphme/em_german_leo_mistral',
        'mistralai/Mistral-Nemo-Instruct-2407',
        'utter-project/EuroLLM-9B-Instruct',
        'mistralai/Mixtral-8x7B-Instruct-v0.1',
        'Qwen/Qwen2.5-14B-Instruct',
    ])
    seen=set(); out=[]
    for p in paths:
        if p not in seen: seen.add(p); out.append(p)
    return out

LLM_PATHS = discover_llm_paths()
print('LLM candidate paths:')
for p in LLM_PATHS:
    tag = '\u2713' if (p.startswith('/') and Path(p).exists()) else 'hub'
    print(f'  [{tag}] {p}')

bnb_cfg = None
if USE_LLM_4BIT:
    try:
        import bitsandbytes  # noqa
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        )
        print('4-bit config ready')
    except ImportError:
        print('  !pip install bitsandbytes  (falling back to fp16 — may OOM for >13B)')

for p in LLM_PATHS:
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        LLM_TOK = AutoTokenizer.from_pretrained(p)
        kwargs = {'device_map': 'auto'}
        if bnb_cfg is not None: kwargs['quantization_config'] = bnb_cfg
        else: kwargs['torch_dtype'] = torch.float16
        LLM = AutoModelForCausalLM.from_pretrained(p, **kwargs)
        LLM.eval()
        mode = '4-bit' if bnb_cfg is not None else 'fp16'
        print(f'\u2705 LLM loaded from {p}  ({mode})')
        break
    except Exception as e:
        print(f'  failed {p}: {type(e).__name__}: {str(e)[:100]}')

if LLM is None:
    LLM_FAILED = True
    print('  \u26a0\ufe0f No LLM loaded — pipeline will fall back to seeds+retrieval only')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'  cuda:{i}  free={free/1e9:.1f}GB / total={total/1e9:.1f}GB')


def llm_generate(prompt, max_new_tokens=900):
    if LLM is None or LLM_FAILED:
        return ''
    try:
        # Use chat template if available
        if hasattr(LLM_TOK, 'apply_chat_template') and LLM_TOK.chat_template:
            msgs = [{'role': 'user', 'content': prompt}]
            inputs = LLM_TOK.apply_chat_template(msgs, return_tensors='pt',
                                                 add_generation_prompt=True)
            inputs = inputs.to(next(LLM.parameters()).device)
            with torch.no_grad():
                out = LLM.generate(inputs, max_new_tokens=max_new_tokens,
                                    do_sample=False, temperature=0.0,
                                    pad_token_id=(LLM_TOK.eos_token_id or 0))
            gen = LLM_TOK.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        else:
            enc = LLM_TOK(prompt, return_tensors='pt', truncation=True, max_length=8192)
            enc = {k: v.to(next(LLM.parameters()).device) for k, v in enc.items()}
            with torch.no_grad():
                out = LLM.generate(**enc, max_new_tokens=max_new_tokens,
                                    do_sample=False, temperature=0.0,
                                    pad_token_id=(LLM_TOK.eos_token_id or 0))
            gen = LLM_TOK.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        return gen
    except Exception as e:
        print(f'  llm_generate exc: {type(e).__name__}: {str(e)[:80]}')
        return ''


In [ ]:
# Cell 12 — LLM-first prediction: retrieval grounding → generate → corpus-validate
RE_ART_Q  = re.compile(r'\bArt(?:icle)?\.?\s*(\d+[a-z]?(?:bis|ter|quater)?)'
                       r'(?:\s*(?:Abs\.|al\.|para\.)\s*(\d+))?'
                       r'(?:\s*(?:lit\.|let\.)\s*([a-z]))?'
                       r'\s+([A-Z][A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)')
RE_BGE_Q  = re.compile(r'\bBGE\s+(\d+)\s+([IVX]+)\s+(\d+)(?:\s+E\.?\s*([\d\.]+))?')
RE_BGER_Q = re.compile(r'\b(\d+[A-Za-z]_\d+/\d+)(?:\s+E\.?\s*([\d\.]+))?')

def extract_seeds(query):
    seeds=[]
    for m in RE_ART_Q.finditer(query):
        art, abs_, lit, c = m.groups()
        parts=[f'Art. {art}']
        if abs_: parts.append(f'Abs. {abs_}')
        if lit:  parts.append(f'lit. {lit}')
        parts.append(c)
        seeds.extend(resolve_against_corpus(' '.join(parts)))
    for m in RE_BGE_Q.finditer(query):
        vol, book, page, e = m.groups()
        base=f'BGE {vol} {book} {page}'
        if e: seeds.extend(resolve_against_corpus(f'{base} E. {e}'))
        seeds.extend(resolve_against_corpus(base))
    for m in RE_BGER_Q.finditer(query):
        case, e = m.groups()
        if e: seeds.extend(resolve_against_corpus(f'{case} E. {e}'))
        seeds.extend(resolve_against_corpus(case))
    return dedup(seeds)

_ALIAS_RE = re.compile(
    r'\b(?:Art(?:icle)?\.?|al\.?)\s*\d+[a-z]?'
    r'(?:\s*(?:Abs\.|al\.|para\.)\s*\d+)?'
    r'(?:\s*(?:lit\.|let\.)\s*[a-z])?'
    r'\s+([A-Z][A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df0-9\.\-]+)')
def expand_query_aliases(q):
    s=q
    for m in _ALIAS_RE.finditer(q):
        c=m.group(1)
        if c in MULTILANG_ABBR: s=s+' '+MULTILANG_ABBR[c]
    return s

def reciprocal_rank_fusion(rankings, weights=None, k=60):
    if weights is None: weights=[1.0]*len(rankings)
    sc={}
    for rk,w in zip(rankings,weights):
        for r,cit in enumerate(rk):
            sc[cit]=sc.get(cit,0.0)+w/(k+r+1)
    return sc

def search_laws(query_en, query_de, top_k=60, bm25_n=300, dense_n=300):
    bm25_q = expand_query_aliases(query_de or query_en)
    bm25_scores = bm25.get_scores(tokenize(bm25_q))
    bm25_top = np.argsort(bm25_scores)[::-1][:bm25_n]
    rankings=[[laws_citations[i] for i in bm25_top]]; weights=[0.3]
    q_en=encode_query(query_en)
    _,I=laws_dense.search(q_en,dense_n)
    rankings.append([laws_citations[i] for i in I[0]]); weights.append(0.6)
    if query_de and query_de!=query_en:
        q_de=encode_query(query_de)
        _,I=laws_dense.search(q_de,dense_n)
        rankings.append([laws_citations[i] for i in I[0]]); weights.append(0.3)
    fused=reciprocal_rank_fusion(rankings,weights=weights)
    return sorted(fused.items(), key=lambda kv:-kv[1])[:top_k]

E_EMB_RECON=None
if USE_COURT and e_dense is not None and hasattr(e_dense,'reconstruct_n'):
    try:
        print('Reconstructing E-level embeddings ...')
        t0=time.time(); E_EMB_RECON=e_dense.reconstruct_n(0,e_dense.ntotal)
        print(f'  {E_EMB_RECON.shape} in {time.time()-t0:.1f}s')
    except Exception as exc:
        print(f'  reconstruct failed: {exc}')

def search_court(query_en, query_de, top_k_j=40, top_k_e=40):
    if not USE_COURT or j_dense is None or e_dense is None: return []
    q_en=encode_query(query_en)
    _,I_en=j_dense.search(q_en,top_k_j)
    sel=set(int(i) for i in I_en[0] if i>=0)
    if query_de and query_de!=query_en:
        q_de=encode_query(query_de)
        _,I_de=j_dense.search(q_de,top_k_j)
        sel.update(int(i) for i in I_de[0] if i>=0)
    e2j=court_meta['e_to_jidx']
    e_idx=[i for i,j in enumerate(e2j) if j in sel]
    if not e_idx or E_EMB_RECON is None: return []
    e_sub=E_EMB_RECON[e_idx]
    s=e_sub@q_en[0]
    if query_de and query_de!=query_en: s=s+e_sub@q_de[0]
    order=np.argsort(-s)[:top_k_e]
    ec=court_meta['e_citations']
    return [(ec[e_idx[k]],float(s[k])) for k in order]

text_by_cit = dict(zip(laws_citations, laws_texts))
if USE_COURT and court_meta.get('e_texts_trunc'):
    for cit,txt in zip(court_meta['e_citations'], court_meta['e_texts_trunc']):
        if cit not in text_by_cit: text_by_cit[cit]=txt
print(f'text_by_cit: {len(text_by_cit):,} entries')

def parse_citations_from_text(text):
    """Extract citation strings from free LLM output, normalize, validate vs corpus."""
    out=[]
    for m in RE_ART_Q.finditer(text):
        art, abs_, lit, c = m.groups()
        parts=[f'Art. {art}']
        if abs_: parts.append(f'Abs. {abs_}')
        if lit:  parts.append(f'lit. {lit}')
        parts.append(c)
        out.extend(resolve_against_corpus(' '.join(parts)))
    for m in RE_BGE_Q.finditer(text):
        vol,book,page,e=m.groups()
        base=f'BGE {vol} {book} {page}'
        if e: out.extend(resolve_against_corpus(f'{base} E. {e}'))
        out.extend(resolve_against_corpus(base))
    for m in RE_BGER_Q.finditer(text):
        case,e=m.groups()
        if e: out.extend(resolve_against_corpus(f'{case} E. {e}'))
        out.extend(resolve_against_corpus(case))
    return dedup(out)

FEWSHOT = (
    "Here are examples of correctly formatted Swiss citation lists. Note the exact format "
    "(German abbreviations, Abs./lit. sub-levels) and that EACH list includes both "
    "substantive law AND the procedural cluster (BGG appeal articles, BV rights) even when "
    "the facts don't mention them.\n\n"
    "EXAMPLE 1 — pre-trial detention, criminal procedure:\n"
    "Art. 221 Abs. 1 StPO\nArt. 222 StPO\nArt. 226 Abs. 1 StPO\nArt. 227 Abs. 1 StPO\n"
    "Art. 228 Abs. 1 StPO\nArt. 229 Abs. 1 StPO\nArt. 5 Abs. 3 BV\nArt. 31 Abs. 1 BV\n"
    "Art. 10 Abs. 2 BV\nArt. 36 BV\nArt. 393 Abs. 1 StPO\nArt. 382 Abs. 1 StPO\n"
    "Art. 428 Abs. 1 StPO\nArt. 78 BGG\nArt. 81 Abs. 1 BGG\nArt. 100 Abs. 1 BGG\n"
    "Art. 42 Abs. 1 BGG\nArt. 42 Abs. 2 BGG\n\n"
    "EXAMPLE 2 — inheritance dispute, private law:\n"
    "Art. 467 ZGB\nArt. 469 Abs. 1 ZGB\nArt. 470 Abs. 1 ZGB\nArt. 471 ZGB\n"
    "Art. 505 Abs. 1 ZGB\nArt. 522 Abs. 1 ZGB\nArt. 457 Abs. 1 ZGB\nArt. 458 Abs. 1 ZGB\n"
    "Art. 8 ZGB\nArt. 18 Abs. 1 OR\nArt. 74 Abs. 1 BGG\nArt. 75 BGG\nArt. 90 BGG\n"
    "Art. 100 Abs. 1 BGG\nArt. 42 Abs. 1 BGG\nArt. 106 Abs. 2 BGG\n\n"
    "EXAMPLE 3 — residential lease termination, contract law:\n"
    "Art. 257 OR\nArt. 257d Abs. 1 OR\nArt. 257d Abs. 2 OR\nArt. 266a Abs. 1 OR\n"
    "Art. 266g Abs. 1 OR\nArt. 271 Abs. 1 OR\nArt. 271a Abs. 1 OR\nArt. 18 Abs. 1 OR\n"
    "Art. 8 ZGB\nArt. 74 Abs. 1 BGG\nArt. 75 BGG\nArt. 90 BGG\nArt. 100 Abs. 1 BGG\n"
    "Art. 42 Abs. 1 BGG\nArt. 106 Abs. 2 BGG\n\n"
)

PROMPT_TMPL = (
    'You are a Swiss legal expert preparing the citation list for a bar-exam style case. '
    'List ALL Swiss federal law provisions and leading cases relevant to the case below. '
    'Include BOTH substantive law (the articles governing the facts) AND procedural law '
    '(appeal provisions like Art. 100 / Art. 42 BGG, procedure codes StPO/ZPO, '
    'constitutional rights BV) typically cited for this kind of dispute. '
    'Aim for 15-30 citations.\n\n'
    + FEWSHOT +
    'Use exact Swiss German citation format, one per line. Output ONLY citations, no commentary.\n\n'
    'CASE:\n{query}\n\n'
    'Some potentially relevant law excerpts (German):\n{grounding}\n\n'
    'Citation list for THIS case (German abbreviations, one per line, 15-30 items):\n'
)

def build_grounding(laws_res, court_res, max_items=40, snippet=160):
    lines=[]
    for cit,_ in laws_res[:max_items]:
        t=text_by_cit.get(cit,'')[:snippet]
        lines.append(f'- {cit}: {t}')
    for cit,_ in court_res[:10]:
        t=text_by_cit.get(cit,'')[:snippet]
        lines.append(f'- {cit}: {t}')
    return '\n'.join(lines)

TOP_N_CAP = 40

def predict(query, qid=None, return_debug=False):
    seeds = extract_seeds(query)
    universal = get_universal()
    query_de = translate_to_de(query) if USE_TRANSLATE else query

    laws_res  = search_laws(query, query_de, top_k=60)
    court_res = search_court(query, query_de, top_k_e=40)
    grounding = build_grounding(laws_res, court_res)

    llm_text = ''
    gen_cits = []
    if LLM is not None and not LLM_FAILED:
        prompt = PROMPT_TMPL.format(query=query[:3000], grounding=grounding[:4000])
        llm_text = llm_generate(prompt, max_new_tokens=900)
        gen_cits = parse_citations_from_text(llm_text)

    # Retrieval fallback pool (used to top up if LLM gave too few)
    retr_pool = dedup([c for c,_ in laws_res] + [c for c,_ in court_res])
    retr_pool = [c for c in retr_pool if not is_sr_numeric(c)]  # drop obscure ordinances

    # Compose: seeds (highest precision) → universal → LLM-generated → retrieval topup
    front = dedup(seeds + universal)
    body = [c for c in gen_cits if c not in set(front)]
    out = dedup(front + body)
    # top up to a reasonable count if LLM produced few
    if len(out) < 15:
        for c in retr_pool:
            if c not in set(out):
                out.append(c)
            if len(out) >= 20: break
    out = granularity_filter(out)[:TOP_N_CAP]

    if return_debug:
        return out, {
            'n_seeds': len(seeds), 'n_gen': len(gen_cits),
            'n_emit': len(out), 'llm_chars': len(llm_text),
            'llm_used': LLM is not None and not LLM_FAILED,
        }
    return out


In [ ]:
# Cell 13 — Val Macro F1 + LLM-generation diagnostic
def per_query_f1(g,p):
    g,p=set(g),set(p)
    if not g and not p: return 1.0
    if not g or not p: return 0.0
    tp=len(g&p)
    if tp==0: return 0.0
    pr=tp/len(p); rc=tp/len(g); return 2*pr*rc/(pr+rc)
def macro_f1(gd,pd_):
    qs=set(gd)|set(pd_)
    return sum(per_query_f1(gd.get(q,[]),pd_.get(q,[])) for q in qs)/max(1,len(qs))

val=pd.read_csv(DATA/'val.csv')
gold={r['query_id']:split_cits(r['gold_citations']) for _,r in val.iterrows()}
print(f'Predicting val (LLM-first) ...')
t0=time.time()
results={}
for _,r in val.iterrows():
    pred,dbg=predict(r['query'], r['query_id'], return_debug=True)
    results[r['query_id']]=(pred,dbg)
print(f'  done in {time.time()-t0:.1f}s')

preds={q:p for q,(p,_) in results.items()}
f1=macro_f1(gold,preds)
print(f'\nVAL Macro F1 = {f1:.4f}\n')

print(f"{'qid':8s} {'F1':>6s} {'gld':>4s} {'emt':>4s} {'tp':>3s} {'seeds':>5s} {'gen':>4s} {'llm_ch':>7s}")
for qid in sorted(gold):
    g=gold[qid]; pred,dbg=results[qid]
    tp=len(set(g)&set(pred))
    print(f"{qid:8s} {per_query_f1(g,pred):6.3f} {len(g):>4d} {len(pred):>4d} {tp:>3d} "
          f"{dbg['n_seeds']:>5d} {dbg['n_gen']:>4d} {dbg['llm_chars']:>7d}")

# show one raw LLM output sample for sanity
print('\n=== sample LLM raw output (val_001) ===')
q0 = val.iloc[0]['query']
if LLM is not None and not LLM_FAILED:
    qd = translate_to_de(q0) if USE_TRANSLATE else q0
    lr = search_laws(q0, qd, top_k=60); cr = search_court(q0, qd, top_k_e=40)
    gr = build_grounding(lr, cr)
    raw = llm_generate(PROMPT_TMPL.format(query=q0[:3000], grounding=gr[:4000]), max_new_tokens=900)
    print(raw[:1500])


In [ ]:
# Cell 14 — submission.csv
test=pd.read_csv(DATA/'test.csv')
rows=[]
print(f'Predicting test ({len(test)} rows) ...')
t0=time.time()
for i,(_,r) in enumerate(test.iterrows(),1):
    cits=predict(r['query'], r['query_id'])
    rows.append({'query_id':r['query_id'],'predicted_citations':';'.join(cits)})
    if i%5==0: print(f'  {i}/{len(test)}  elapsed {time.time()-t0:.1f}s', flush=True)
print(f'  done in {time.time()-t0:.1f}s')

sub=pd.DataFrame(rows)
out=WORK/'submission.csv'
sub.to_csv(out,index=False)
print(f'\nWrote {out}  rows={len(sub)}')
print(sub.head(3).to_string(index=False))
counts=sub['predicted_citations'].str.count(';').add(1)
print(f'\nemit count  min={counts.min()}  mean={counts.mean():.1f}  max={counts.max()}')
allp=[]
for cs in sub['predicted_citations']: allp.extend(cs.split(';'))
uniq=set(allp)
inc=sum(1 for c in uniq if c in CORPUS)
print(f'corpus coverage: {inc}/{len(uniq)} = {inc/max(1,len(uniq)):.4f}')
